# 12 · Graph v2 experiments — label-aware patient graphs vs. XGBoost

Self-contained: every repo module this needs is written out by the
`%%writefile` cells below (built from the repo by `scripts/nbmake.py`'s
`#%% embed` marker), and the MIMIC-IV mirror is fetched with `kagglehub`, so
this runs with **no secrets and no manual dataset attach**.

What one run does, for every seed in `SEEDS` (same 60/20/20 split and same
model_val/gate_selection/gate_certification cut as notebooks 10/11):

1. **XGBoost references** on the same split -- the phase-1 plain config, and
   a tuned native-NaN config early-stopped on `model_val`.
2. **Corrected Arch4R `encoder_only` baseline** -- the phase-1 arm trained
   one full-batch step per epoch (5 steps total at 5 epochs); here it is
   mini-batched, i.e. what the "graph signal" criterion should have been
   compared against.
3. **Encoder preprocessing ablation** -- the Arch4R 1.5-IQR winsorization vs.
   0.5/99.5-percentile clipping vs. a train-fit quantile transform, each with
   and without per-feature missingness indicators.
4. **Label-homophily diagnostics** of a feature kNN patient graph.
5. **G1 -- TabR-style retrieval** over a learned metric: each patient
   attends over its m nearest *training* patients (self excluded) and reads
   their labels + key differences. Exit 0 = graph-free, exit 1 = retrieved.
6. **G2 -- masked-label PPR diffusion** (Arch4R lineage): a learned kNN graph
   over train patients, APPNP-style personalized-PageRank diffusion of both
   embeddings and *masked* training labels (label trick: a scored patient's
   own label is never an input), one exit per diffusion checkpoint.

Model selection uses `model_val` only; test is read once per config, for
reporting. Results -> `/kaggle/working/results_v2/`.

In [ ]:
import os, sys, json, time, math, glob, subprocess
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

WORK = "/kaggle/working"
os.makedirs(f"{WORK}/src", exist_ok=True)
open(f"{WORK}/src/__init__.py", "w").close()
sys.path.insert(0, WORK)
OUT = f"{WORK}/results_v2"
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [42]
print(f"[v2] device={DEV} torch={torch.__version__}", flush=True)

In [2]:
%%writefile /kaggle/working/src/mimic_concepts.py
"""Clinical concept dictionaries for MIMIC-IV v3.1.

Everything that maps a raw MIMIC identifier to a clinical concept lives here so
`mimic_iv.py` stays pure ETL mechanics. Three groups:

  * `CHART_ITEMS` / `LAB_ITEMS`  -- itemid -> concept, for the streaming
    aggregators over `icu/chartevents.csv` and `hosp/labevents.csv`.
  * `INPUT_ITEMS` / `PROC_ITEMS` / `OUTPUT_ITEMS` -- ICU treatment exposure,
    the "interventions and medications" block of the SBSCGM feature encoder
    (arXiv:2508.00615 sec. 3.2).
  * `CHARLSON_ICD9` / `CHARLSON_ICD10` -- Quan et al. (2005) code-prefix maps
    for the comorbidity block.

Physiologic plausibility ranges are attached to every numeric concept.
MIMIC chart/lab values contain transcription errors (a heart rate of 8000, a
creatinine of -1); clipping to a clinical range before aggregation is standard
practice and stops one bad row from destroying a patient's min/max features.
"""

# --------------------------------------------------------------- ICU vitals
# itemid -> (concept, unit_hint, lo, hi). Both invasive and non-invasive
# pressures map to the same concept: which line a patient has is a treatment
# decision, not a different physiologic quantity.
CHART_ITEMS = {
    220045: ("heart_rate", "bpm", 10, 300),
    220050: ("sbp", "mmHg", 20, 300),
    220179: ("sbp", "mmHg", 20, 300),
    225309: ("sbp", "mmHg", 20, 300),
    220051: ("dbp", "mmHg", 5, 200),
    220180: ("dbp", "mmHg", 5, 200),
    225310: ("dbp", "mmHg", 5, 200),
    220052: ("map", "mmHg", 10, 250),
    220181: ("map", "mmHg", 10, 250),
    225312: ("map", "mmHg", 10, 250),
    220210: ("resp_rate", "bpm", 2, 80),
    224690: ("resp_rate", "bpm", 2, 80),
    220277: ("spo2", "%", 20, 100),
    223761: ("temp_f", "F", 70, 115),
    223762: ("temp_c", "C", 20, 46),
    220739: ("gcs_eye", "pts", 1, 4),
    223900: ("gcs_verbal", "pts", 1, 5),
    223901: ("gcs_motor", "pts", 1, 6),
    220621: ("glucose_chart", "mg/dL", 10, 1500),
    225664: ("glucose_chart", "mg/dL", 10, 1500),
    226512: ("weight_kg", "kg", 20, 400),
    223835: ("fio2", "%", 20, 100),
}

# ------------------------------------------------------------------- labs
LAB_ITEMS = {
    50912: ("creatinine", "mg/dL", 0.1, 25),
    51006: ("bun", "mg/dL", 1, 250),
    50971: ("potassium", "mEq/L", 1, 12),
    50983: ("sodium", "mEq/L", 90, 200),
    50902: ("chloride", "mEq/L", 60, 160),
    50882: ("bicarbonate", "mEq/L", 3, 60),
    50931: ("glucose_lab", "mg/dL", 10, 1500),
    51221: ("hematocrit", "%", 5, 70),
    51222: ("hemoglobin", "g/dL", 2, 25),
    51301: ("wbc", "K/uL", 0.1, 200),
    51265: ("platelets", "K/uL", 5, 1500),
    50813: ("lactate", "mmol/L", 0.2, 30),
    50820: ("ph", "", 6.5, 8.0),
    50821: ("po2", "mmHg", 10, 700),
    50818: ("pco2", "mmHg", 5, 150),
    50802: ("base_excess", "mEq/L", -30, 30),
    51237: ("inr", "", 0.5, 15),
    51275: ("ptt", "s", 10, 200),
    50885: ("bilirubin", "mg/dL", 0.1, 50),
    50862: ("albumin", "g/dL", 0.5, 7),
    50863: ("alk_phos", "IU/L", 10, 2000),
    50861: ("alt", "IU/L", 5, 5000),
    50878: ("ast", "IU/L", 5, 5000),
    50960: ("magnesium", "mg/dL", 0.5, 8),
    50893: ("calcium", "mg/dL", 3, 20),
    50970: ("phosphate", "mg/dL", 0.5, 20),
    51003: ("troponin_t", "ng/mL", 0.0, 30),
    50889: ("crp", "mg/L", 0.0, 400),
}

# --------------------------------------------------- ICU treatment exposure
# Binary "did this patient receive X in the window" flags. These are the
# strongest single predictors in an ICU cohort and are exactly the
# "high-risk intervention" indicators the SBSCGM encoder calls for.
INPUT_ITEMS = {
    221906: "norepinephrine",
    221289: "epinephrine",
    221749: "phenylephrine",
    222315: "vasopressin",
    221662: "dopamine",
    221653: "dobutamine",
    221468: "milrinone",
    222168: "propofol",
    221668: "midazolam",
    221744: "fentanyl",
    225158: "crystalloid_ns",
    225828: "crystalloid_lr",
    220862: "albumin_infusion",
    225168: "prbc_transfusion",
}

# Vasopressors get their own aggregate flag: any of these means shock.
VASOPRESSORS = ("norepinephrine", "epinephrine", "phenylephrine",
                "vasopressin", "dopamine")

PROC_ITEMS = {
    225792: "invasive_ventilation",
    225794: "noninvasive_ventilation",
    227194: "extubation",
    224385: "intubation",
    225802: "crrt",
    225803: "crrt",
    225809: "crrt",
    225955: "crrt",
    225441: "hemodialysis",
    225805: "peritoneal_dialysis",
    224263: "central_line",
    224264: "central_line",
    224267: "arterial_line",
    225752: "arterial_line",
    225454: "chest_tube",
}

# Urine output is charted under a long tail of itemids (Foley, void, ileoconduit
# ...). All of them are summed into a single hourly rate.
# Urine output. 227489 is "GU Irrigant/Urine Volume Out", which includes the
# irrigant that was instilled; 227488 is that instilled volume ("GU Irrigant
# Volume In") and is therefore *subtracted*, matching MIMIC's own
# `urine_output` concept. Summing 227488 as if it were urine inflates output
# for anyone on bladder irrigation.
OUTPUT_URINE_ITEMS = (
    226559, 226560, 226561, 226584, 226563, 226564, 226565,
    226567, 226557, 226558, 227489,
)
OUTPUT_URINE_NEGATIVE_ITEMS = (227488,)

# A single charted void or bag reading above this is a charting error, not a
# measurement. The derived rate is bounded separately in `mimic_iv`.
URINE_EVENT_MAX_ML = 2500.0
# Sustained output above this over a whole observation window is not
# physiologic (brisk post-obstructive diuresis peaks well below it).
URINE_RATE_MAX_ML_PER_HR = 1000.0

# ------------------------------------------------------------- comorbidities
# Quan et al. (2005) ICD code-prefix mapping for the Charlson index. Matched on
# `icd_code.startswith(prefix)` after stripping dots, which is how MIMIC-IV
# stores codes.
CHARLSON_ICD9 = {
    "myocardial_infarction": ("410", "412"),
    "congestive_heart_failure": ("39891", "40201", "40211", "40291", "40401",
                                 "40403", "40411", "40413", "40491", "40493",
                                 "4254", "4255", "4257", "4258", "4259", "428"),
    "peripheral_vascular": ("0930", "4373", "440", "441", "4431", "4432",
                            "4438", "4439", "4471", "5571", "5579", "V434"),
    "cerebrovascular": ("36234", "430", "431", "432", "433", "434", "435",
                        "436", "437", "438"),
    "dementia": ("290", "2941", "3312"),
    "chronic_pulmonary": ("4168", "4169", "490", "491", "492", "493", "494",
                          "495", "496", "500", "501", "502", "503", "504",
                          "505", "5064", "5081", "5088"),
    "rheumatic": ("4465", "7100", "7101", "7102", "7103", "7104", "7140",
                  "7141", "7142", "7148", "725"),
    "peptic_ulcer": ("531", "532", "533", "534"),
    "mild_liver_disease": ("07022", "07023", "07032", "07033", "07044",
                           "07054", "0706", "0709", "570", "571", "5733",
                           "5734", "5738", "5739", "V427"),
    "diabetes_uncomplicated": ("2500", "2501", "2502", "2503", "2508", "2509"),
    "diabetes_complicated": ("2504", "2505", "2506", "2507"),
    "hemiplegia": ("3341", "342", "343", "3440", "3441", "3442", "3443",
                   "3444", "3445", "3446", "3449"),
    "renal_disease": ("40301", "40311", "40391", "40402", "40403", "40412",
                      "40413", "40492", "40493", "582", "5830", "5831",
                      "5832", "5834", "5836", "5837", "585", "586", "5880",
                      "V420", "V451", "V56"),
    "malignancy": ("140", "141", "142", "143", "144", "145", "146", "147",
                   "148", "149", "150", "151", "152", "153", "154", "155",
                   "156", "157", "158", "159", "160", "161", "162", "163",
                   "164", "165", "17", "18", "190", "191", "192", "193",
                   "194", "195", "200", "201", "202", "203", "204", "205",
                   "206", "207", "208", "2386"),
    "severe_liver_disease": ("4560", "4561", "4562", "5722", "5723", "5724",
                             "5728"),
    "metastatic_cancer": ("196", "197", "198", "199"),
    "hiv": ("042", "043", "044"),
}

CHARLSON_ICD10 = {
    "myocardial_infarction": ("I21", "I22", "I252"),
    "congestive_heart_failure": ("I099", "I110", "I130", "I132", "I255",
                                 "I420", "I425", "I426", "I427", "I428",
                                 "I429", "I43", "I50", "P290"),
    "peripheral_vascular": ("I70", "I71", "I731", "I738", "I739", "I771",
                            "I790", "I792", "K551", "K558", "K559", "Z958",
                            "Z959"),
    "cerebrovascular": ("G45", "G46", "H340", "I60", "I61", "I62", "I63",
                        "I64", "I65", "I66", "I67", "I68", "I69"),
    "dementia": ("F00", "F01", "F02", "F03", "F051", "G30", "G311"),
    "chronic_pulmonary": ("I278", "I279", "J40", "J41", "J42", "J43", "J44",
                          "J45", "J46", "J47", "J60", "J61", "J62", "J63",
                          "J64", "J65", "J66", "J67", "J684", "J701", "J703"),
    "rheumatic": ("M05", "M06", "M315", "M32", "M33", "M34", "M351", "M353",
                  "M360"),
    "peptic_ulcer": ("K25", "K26", "K27", "K28"),
    "mild_liver_disease": ("B18", "K700", "K701", "K702", "K703", "K709",
                           "K713", "K714", "K715", "K717", "K73", "K74",
                           "K760", "K762", "K763", "K764", "K768", "K769",
                           "Z944"),
    "diabetes_uncomplicated": ("E100", "E101", "E106", "E108", "E109",
                               "E110", "E111", "E116", "E118", "E119",
                               "E120", "E121", "E126", "E128", "E129",
                               "E130", "E131", "E136", "E138", "E139",
                               "E140", "E141", "E146", "E148", "E149"),
    "diabetes_complicated": ("E102", "E103", "E104", "E105", "E107", "E112",
                             "E113", "E114", "E115", "E117", "E122", "E123",
                             "E124", "E125", "E127", "E132", "E133", "E134",
                             "E135", "E137", "E142", "E143", "E144", "E145",
                             "E147"),
    "hemiplegia": ("G041", "G114", "G801", "G802", "G81", "G82", "G830",
                   "G831", "G832", "G833", "G834", "G839"),
    "renal_disease": ("I120", "I131", "N032", "N033", "N034", "N035", "N036",
                      "N037", "N052", "N053", "N054", "N055", "N056", "N057",
                      "N18", "N19", "N250", "Z490", "Z491", "Z492", "Z940",
                      "Z992"),
    "malignancy": ("C0", "C1", "C2", "C30", "C31", "C32", "C33", "C34",
                   "C37", "C38", "C39", "C40", "C41", "C43", "C45", "C46",
                   "C47", "C48", "C49", "C50", "C51", "C52", "C53", "C54",
                   "C55", "C56", "C57", "C58", "C60", "C61", "C62", "C63",
                   "C64", "C65", "C66", "C67", "C68", "C69", "C70", "C71",
                   "C72", "C73", "C74", "C75", "C76", "C81", "C82", "C83",
                   "C84", "C85", "C88", "C90", "C91", "C92", "C93", "C94",
                   "C95", "C96", "C97"),
    "severe_liver_disease": ("I850", "I859", "I864", "I982", "K704", "K711",
                             "K721", "K729", "K765", "K766", "K767"),
    "metastatic_cancer": ("C77", "C78", "C79", "C80"),
    "hiv": ("B20", "B21", "B22", "B24"),
}

# Weights from the original Charlson index; used for the scalar summary score.
CHARLSON_WEIGHTS = {
    "myocardial_infarction": 1, "congestive_heart_failure": 1,
    "peripheral_vascular": 1, "cerebrovascular": 1, "dementia": 1,
    "chronic_pulmonary": 1, "rheumatic": 1, "peptic_ulcer": 1,
    "mild_liver_disease": 1, "diabetes_uncomplicated": 1,
    "diabetes_complicated": 2, "hemiplegia": 2, "renal_disease": 2,
    "malignancy": 2, "severe_liver_disease": 3, "metastatic_cancer": 6,
    "hiv": 6,
}

COMORBIDITIES = tuple(CHARLSON_WEIGHTS)

# ------------------------------------------------------- categorical binning
# MIMIC-IV race has ~30 levels, most of them sparse sub-categories. Collapsing
# to the OMB roll-up keeps the one-hot block small and the cell counts usable.
RACE_GROUPS = {
    "WHITE": "white", "BLACK": "black", "HISPANIC": "hispanic",
    "ASIAN": "asian", "AMERICAN INDIAN": "other", "NATIVE HAWAIIAN": "other",
    "OTHER": "other", "UNKNOWN": "unknown", "UNABLE TO OBTAIN": "unknown",
    "PATIENT DECLINED": "unknown", "PORTUGUESE": "other",
    "SOUTH AMERICAN": "hispanic", "MULTIPLE": "other",
}

ADMISSION_TYPES = (
    "EW EMER.", "OBSERVATION ADMIT", "URGENT", "EU OBSERVATION",
    "ELECTIVE", "SURGICAL SAME DAY ADMISSION", "DIRECT EMER.",
    "DIRECT OBSERVATION", "AMBULATORY OBSERVATION",
)

CARE_UNITS = (
    "Medical Intensive Care Unit (MICU)",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)",
    "Surgical Intensive Care Unit (SICU)",
    "Cardiac Vascular Intensive Care Unit (CVICU)",
    "Coronary Care Unit (CCU)",
    "Trauma SICU (TSICU)",
    "Neuro Intermediate", "Neuro Stepdown", "Neuro Surgical Intensive Care Unit (Neuro SICU)",
)


def race_group(value):
    """Collapse a raw MIMIC race string to its OMB roll-up."""
    if not isinstance(value, str):
        return "unknown"
    up = value.upper()
    for prefix, group in RACE_GROUPS.items():
        if up.startswith(prefix):
            return group
    return "other"


def charlson_groups(icd_code, icd_version):
    """Comorbidity groups a single ICD code belongs to (usually 0 or 1)."""
    code = str(icd_code).replace(".", "").strip().upper()
    table = CHARLSON_ICD9 if int(icd_version) == 9 else CHARLSON_ICD10
    return [g for g, prefixes in table.items() if code.startswith(prefixes)]

Writing /kaggle/working/src/mimic_concepts.py


In [3]:
%%writefile /kaggle/working/src/mimic_iv.py
"""Native ETL for the Kaggle `akshaybe/updated-mimic-iv` mirror of MIMIC-IV v3.1.

The mirror ships PhysioNet's directory layout verbatim:

    data/raw/mimic-iv-3.1/hosp/{patients,admissions,diagnoses_icd,labevents,...}.csv
    data/raw/mimic-iv-3.1/icu/{icustays,chartevents,inputevents,...}.csv

This module turns those tables into the single patient-level design matrix the
graph pipeline consumes. Nothing downstream knows about MIMIC schemas.

Cohort
------
Adult (>= 18) patients, **first ICU stay only** (so one row per `subject_id`,
which makes the patient-level split trivially exact), stay length at least
`min_los_hours`. Features come from a fixed observation window
`[intime, intime + obs_window_hours)`; the label is in-hospital mortality.

Leakage controls, all asserted in `build_cohort`
------------------------------------------------
1. **Window closure.** Every event used is timestamped strictly inside the
   observation window. No aggregate reaches past `intime + obs_window_hours`.
2. **Deaths inside the window are dropped.** A patient who dies at hour 12 of a
   24 h window has their outcome partly *inside* their own feature window;
   keeping them makes the task trivially easy for the wrong reason.
3. **Comorbidities come from prior admissions only.** `diagnoses_icd` holds
   billing codes assigned at *discharge*, so the current admission's codes
   encode the outcome. Only admissions whose `dischtime` precedes `intime`
   contribute. This is the single biggest leak in naive MIMIC pipelines.
4. **One stay per patient**, so no patient can straddle two splits.

Feature blocks
--------------
Blocks mirror the SBSCGM patient encoder (arXiv:2508.00615 sec. 3.2) and are
carried through to graph construction as *views*:

    demographics  age, sex, admission route, care unit, insurance, race
    comorbidity   Charlson binary groups + weighted index, from history
    labs          first-window aggregates over `hosp/labevents`
    vitals        first-window aggregates over `icu/chartevents`
    treatment     intervention exposure from inputevents/procedureevents,
                  plus urine output rate

Whichever source tables are absent on disk simply contribute no block --
`scripts/fetch_mimic.py` downloads them in tiers, and a core-only checkout still
produces a usable (if smaller) cohort.
"""

import hashlib
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from src import mimic_concepts as mc

# Aggregates computed per numeric concept inside the observation window. `first`
# and `last` together carry the trend, which is what a static snapshot loses --
# the criticism levelled at the ICU-LoS similarity-graph work in the review.
NUMERIC_AGGS = ("first", "last", "min", "max", "mean", "count")


# ------------------------------------------------------------------ plumbing

def find_root(base="data/raw"):
    """Locate the MIMIC-IV tree under `base`. Returns a path or None."""
    for candidate in (
        os.path.join(base, "mimic-iv-3.1"),
        os.path.join(base, "mimic-iv"),
        base,
    ):
        if os.path.isdir(os.path.join(candidate, "hosp")):
            return candidate
    return None


def table_path(root, rel):
    """Path to a table, tolerating the gzipped variant. None if absent."""
    for suffix in ("", ".gz"):
        path = os.path.join(root, rel + suffix)
        if os.path.exists(path):
            return path
    return None


def has_table(root, rel):
    return table_path(root, rel) is not None


def read_table(root, rel, **kwargs):
    path = table_path(root, rel)
    if path is None:
        raise FileNotFoundError(f"{rel} not found under {root}")
    return pd.read_csv(path, **kwargs)


def _duckdb():
    """DuckDB connection, or None. Used to scan the multi-GB event tables."""
    try:
        import duckdb
    except ImportError:
        return None
    con = duckdb.connect()
    con.execute("PRAGMA threads=4")
    return con


# -------------------------------------------------------------------- cohort

def _base_cohort(root, cfg):
    """patients x admissions x icustays -> one row per included ICU stay."""
    d = cfg["data"]
    window_h = float(d["obs_window_hours"])
    min_los_h = float(d.get("min_los_hours", window_h + 4.0))

    patients = read_table(
        root, "hosp/patients.csv",
        usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
    )
    admissions = read_table(
        root, "hosp/admissions.csv",
        usecols=["subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
                 "admission_type", "admission_location", "insurance",
                 "marital_status", "race", "hospital_expire_flag"],
        parse_dates=["admittime", "dischtime", "deathtime"],
    )
    stays = read_table(
        root, "icu/icustays.csv",
        usecols=["subject_id", "hadm_id", "stay_id", "first_careunit",
                 "intime", "outtime", "los"],
        parse_dates=["intime", "outtime"],
    )

    n_stays_raw = len(stays)
    # First ICU stay per patient, in time order.
    stays = stays.sort_values(["subject_id", "intime"])
    stays = stays.drop_duplicates("subject_id", keep="first")

    df = stays.merge(admissions, on=["subject_id", "hadm_id"], how="inner")
    df = df.merge(patients, on="subject_id", how="inner")

    # MIMIC-IV date-shifts each patient into an `anchor_year`; age at admission
    # is anchor_age plus the years elapsed since that anchor.
    df["age"] = df["anchor_age"] + (df["admittime"].dt.year - df["anchor_year"])

    steps = [("icu stays", n_stays_raw), ("first stay per patient", len(df))]

    df = df[df["age"] >= 18]
    steps.append(("adult", len(df)))

    df["los_hours"] = (df["outtime"] - df["intime"]).dt.total_seconds() / 3600.0
    df = df[df["los_hours"] >= min_los_h]
    steps.append((f"los >= {min_los_h:.0f}h", len(df)))

    # Leakage control 2: the outcome must fall strictly after the feature window.
    df["window_end"] = df["intime"] + pd.Timedelta(hours=window_h)
    died_in_window = df["deathtime"].notna() & (df["deathtime"] <= df["window_end"])
    df = df[~died_in_window]
    steps.append(("died after window", len(df)))

    df = df[df["dischtime"] > df["window_end"]]
    steps.append(("discharged after window", len(df)))

    df = df.reset_index(drop=True)
    df["label"] = df["hospital_expire_flag"].astype(np.int64)
    return df, steps


# ------------------------------------------------------------ feature blocks

def _demographics_block(df):
    """Always-observed context features. These anchor the similarity graph."""
    out = pd.DataFrame(index=df.index)
    out["age"] = df["age"].astype(np.float32)
    out["is_female"] = (df["gender"] == "F").astype(np.float32)
    out["los_planned_elective"] = df["admission_type"].isin(
        ("ELECTIVE", "SURGICAL SAME DAY ADMISSION")
    ).astype(np.float32)
    out["admitted_via_ed"] = df["admission_location"].fillna("").str.contains(
        "EMERGENCY"
    ).astype(np.float32)

    for adm in mc.ADMISSION_TYPES:
        key = adm.lower().replace(" ", "_").replace(".", "")
        out[f"adm_{key}"] = (df["admission_type"] == adm).astype(np.float32)
    for unit in mc.CARE_UNITS:
        key = unit.split("(")[-1].strip(")").lower().replace("/", "_").replace(" ", "_")
        out[f"unit_{key}"] = (df["first_careunit"] == unit).astype(np.float32)

    race = df["race"].map(mc.race_group)
    for group in sorted(set(mc.RACE_GROUPS.values())):
        out[f"race_{group}"] = (race == group).astype(np.float32)
    for ins in ("Medicare", "Medicaid", "Private", "Other"):
        out[f"insurance_{ins.lower()}"] = (df["insurance"] == ins).astype(np.float32)
    out["married"] = (df["marital_status"] == "MARRIED").astype(np.float32)
    return out


def _comorbidity_block(root, df):
    """Charlson groups from *prior* admissions only (leakage control 3)."""
    cols = [f"cci_{g}" for g in mc.COMORBIDITIES] + ["cci_score", "n_prior_admissions"]
    if not has_table(root, "hosp/diagnoses_icd.csv"):
        return pd.DataFrame(0.0, index=df.index, columns=cols, dtype=np.float32)

    diag = read_table(root, "hosp/diagnoses_icd.csv",
                      usecols=["subject_id", "hadm_id", "icd_code", "icd_version"])
    diag = diag[diag["subject_id"].isin(set(df["subject_id"]))]

    # Attach each diagnosis to its admission's discharge time, then keep only
    # admissions that closed before this patient's ICU window opened.
    adm_times = read_table(root, "hosp/admissions.csv",
                           usecols=["subject_id", "hadm_id", "dischtime"],
                           parse_dates=["dischtime"])
    diag = diag.merge(adm_times, on=["subject_id", "hadm_id"], how="left")
    diag = diag.merge(df[["subject_id", "intime"]], on="subject_id", how="inner")
    diag = diag[diag["dischtime"] < diag["intime"]]

    # Map every distinct (code, version) once -- there are ~2e4 of them versus
    # millions of rows.
    keys = diag[["icd_code", "icd_version"]].drop_duplicates()
    lookup = {
        (row.icd_code, row.icd_version): mc.charlson_groups(row.icd_code, row.icd_version)
        for row in keys.itertuples()
    }

    flags = {g: set() for g in mc.COMORBIDITIES}
    for subj, code, ver in zip(diag["subject_id"], diag["icd_code"], diag["icd_version"]):
        for group in lookup[(code, ver)]:
            flags[group].add(subj)

    out = pd.DataFrame(index=df.index)
    score = np.zeros(len(df), dtype=np.float32)
    for group in mc.COMORBIDITIES:
        hit = df["subject_id"].isin(flags[group]).to_numpy(dtype=np.float32)
        out[f"cci_{group}"] = hit
        score += mc.CHARLSON_WEIGHTS[group] * hit
    out["cci_score"] = score

    prior = diag.groupby("subject_id")["hadm_id"].nunique()
    out["n_prior_admissions"] = (
        df["subject_id"].map(prior).fillna(0.0).to_numpy(dtype=np.float32)
    )
    return out


def _window_sql(source, id_col, time_col, value_col, item_col, itemids):
    """SQL that aggregates one event table inside each stay's window."""
    ids = ", ".join(str(i) for i in itemids)
    return f"""
        SELECT w.row_id, e.{item_col} AS itemid,
               min_by(e.{value_col}, e.{time_col}) AS v_first,
               max_by(e.{value_col}, e.{time_col}) AS v_last,
               min(e.{value_col})   AS v_min,
               max(e.{value_col})   AS v_max,
               avg(e.{value_col})   AS v_mean,
               count(e.{value_col}) AS v_count
        FROM read_csv_auto('{source}', ignore_errors=true) e
        JOIN win w ON e.{id_col} = w.{id_col}
        WHERE e.{item_col} IN ({ids})
          AND e.{value_col} IS NOT NULL
          AND e.{time_col} >= w.intime
          AND e.{time_col} <  w.window_end
        GROUP BY w.row_id, e.{item_col}
    """


def _aggregate_events(root, df, rel, id_col, time_col, value_col, items,
                      prefix, chunksize=4_000_000, verbose=True):
    """Window-aggregate a large event table into per-concept feature columns.

    Uses DuckDB when it is importable (it scans the 18-42 GB CSVs with predicate
    pushdown and constant memory); otherwise falls back to a chunked pandas
    scan, which is slower but has no extra dependency.
    """
    concepts = sorted({name for name, *_ in items.values()})
    ranges = {name: (lo, hi) for name, _, lo, hi in items.values()}
    columns = [f"{prefix}_{c}_{a}" for c in concepts for a in NUMERIC_AGGS]

    path = table_path(root, rel)
    if path is None:
        if verbose:
            print(f"  [{prefix}] {rel} absent -- block skipped")
        return pd.DataFrame(np.nan, index=df.index, columns=columns, dtype=np.float32)

    win = df[[id_col, "intime", "window_end"]].copy()
    win["row_id"] = np.arange(len(df))

    con = _duckdb()
    if con is not None:
        if verbose:
            print(f"  [{prefix}] duckdb scan of {rel} ...", flush=True)
        con.register("win", win)
        agg = con.execute(
            _window_sql(path, id_col, time_col, value_col, "itemid", sorted(items))
        ).fetch_df()
        con.close()
    else:
        agg = _aggregate_events_pandas(path, win, id_col, time_col, value_col,
                                       items, chunksize, verbose, prefix)

    out = pd.DataFrame(np.nan, index=df.index, columns=columns, dtype=np.float32)
    if len(agg) == 0:
        return out

    agg["concept"] = agg["itemid"].map({k: v[0] for k, v in items.items()})
    # One itemid per concept per stay is the common case, but pressures and
    # glucose have several sources; combine them the way each aggregate implies.
    grouped = agg.groupby(["row_id", "concept"]).agg(
        v_first=("v_first", "first"), v_last=("v_last", "last"),
        v_min=("v_min", "min"), v_max=("v_max", "max"),
        v_mean=("v_mean", "mean"), v_count=("v_count", "sum"),
    ).reset_index()

    for concept, block in grouped.groupby("concept"):
        lo, hi = ranges[concept]
        rows = block["row_id"].to_numpy()
        for agg_name in NUMERIC_AGGS:
            vals = block[f"v_{agg_name}"].to_numpy(dtype=np.float64)
            if agg_name != "count":
                vals = np.where((vals < lo) | (vals > hi), np.nan, vals)
            # `out` is float32; assigning float64 into it is deprecated in
            # pandas and becomes an error in a future version.
            out.iloc[rows, out.columns.get_loc(f"{prefix}_{concept}_{agg_name}")] = \
                vals.astype(np.float32)

    # A concept that was never measured has count 0, not "missing count".
    for concept in concepts:
        col = f"{prefix}_{concept}_count"
        out[col] = out[col].fillna(0.0)
    return out.astype(np.float32)


def _aggregate_events_pandas(path, win, id_col, time_col, value_col, items,
                             chunksize, verbose, prefix):
    """Chunked fallback for `_aggregate_events` when DuckDB is unavailable."""
    keep_ids = set(win[id_col])
    bounds = win.set_index(id_col)[["intime", "window_end", "row_id"]]
    parts = []
    reader = pd.read_csv(
        path, usecols=[id_col, time_col, value_col, "itemid"],
        parse_dates=[time_col], chunksize=chunksize,
        dtype={value_col: np.float64}, on_bad_lines="skip",
    )
    for i, chunk in enumerate(reader):
        chunk = chunk[chunk["itemid"].isin(items) & chunk[id_col].isin(keep_ids)]
        chunk = chunk.dropna(subset=[value_col])
        if chunk.empty:
            continue
        chunk = chunk.join(bounds, on=id_col)
        chunk = chunk[(chunk[time_col] >= chunk["intime"])
                      & (chunk[time_col] < chunk["window_end"])]
        if not chunk.empty:
            parts.append(chunk[["row_id", "itemid", time_col, value_col]])
        if verbose and i % 10 == 0:
            print(f"  [{prefix}] chunk {i}, {sum(len(p) for p in parts):,} rows kept",
                  flush=True)

    if not parts:
        return pd.DataFrame(columns=["row_id", "itemid", "v_first", "v_last",
                                     "v_min", "v_max", "v_mean", "v_count"])
    events = pd.concat(parts).sort_values(time_col)
    return events.groupby(["row_id", "itemid"])[value_col].agg(
        v_first="first", v_last="last", v_min="min", v_max="max",
        v_mean="mean", v_count="count",
    ).reset_index()


def _treatment_block(root, df, verbose=True):
    """Binary intervention exposure inside the window, plus urine output rate.

    These are the "interventions and medications" features of the SBSCGM
    encoder: whether a patient was on pressors, ventilated, or dialysed in the
    first window is both highly prognostic and cheap to extract.
    """
    names = sorted(set(mc.INPUT_ITEMS.values()) | set(mc.PROC_ITEMS.values()))
    columns = [f"tx_{n}" for n in names] + ["tx_any_vasopressor", "tx_n_interventions",
                                            "tx_urine_ml_per_hr"]
    out = pd.DataFrame(0.0, index=df.index, columns=columns, dtype=np.float32)

    win = df[["stay_id", "intime", "window_end"]].copy()
    win["row_id"] = np.arange(len(df))
    window_h = (df["window_end"] - df["intime"]).dt.total_seconds().to_numpy() / 3600.0

    sources = [
        ("icu/inputevents.csv", "starttime", mc.INPUT_ITEMS),
        ("icu/procedureevents.csv", "starttime", mc.PROC_ITEMS),
    ]
    for rel, time_col, mapping in sources:
        path = table_path(root, rel)
        if path is None:
            if verbose:
                print(f"  [tx] {rel} absent -- block partially skipped")
            continue
        hits = _exposure_rows(path, win, time_col, sorted(mapping), verbose, rel)
        for itemid, rows in hits.items():
            col = f"tx_{mapping[itemid]}"
            out.iloc[list(rows), out.columns.get_loc(col)] = 1.0

    out["tx_any_vasopressor"] = out[
        [f"tx_{v}" for v in mc.VASOPRESSORS if f"tx_{v}" in out]
    ].max(axis=1)
    out["tx_n_interventions"] = out[[f"tx_{n}" for n in names]].sum(axis=1)

    urine_path = table_path(root, "icu/outputevents.csv")
    if urine_path is not None:
        totals = _urine_totals(urine_path, win, verbose)
        rate = np.zeros(len(df), dtype=np.float32)
        rate[list(totals)] = np.array(list(totals.values()), dtype=np.float32)
        rate = rate / np.maximum(window_h, 1e-6)
        # Irrigant subtraction can drive a stay negative; both that and a
        # supraphysiologic rate mean "not measurable", which is missing, not 0.
        out["tx_urine_ml_per_hr"] = np.where(
            (rate < 0) | (rate > mc.URINE_RATE_MAX_ML_PER_HR), np.nan, rate)
    else:
        out["tx_urine_ml_per_hr"] = np.nan
    return out


def _exposure_rows(path, win, time_col, itemids, verbose, rel):
    """{itemid: set(row_id)} for events starting inside each stay's window."""
    con = _duckdb()
    ids = ", ".join(str(i) for i in itemids)
    if con is not None:
        if verbose:
            print(f"  [tx] duckdb scan of {rel} ...", flush=True)
        con.register("win", win)
        res = con.execute(f"""
            SELECT DISTINCT e.itemid, w.row_id
            FROM read_csv_auto('{path}', ignore_errors=true) e
            JOIN win w ON e.stay_id = w.stay_id
            WHERE e.itemid IN ({ids})
              AND e.{time_col} >= w.intime AND e.{time_col} < w.window_end
        """).fetch_df()
        con.close()
    else:
        bounds = win.set_index("stay_id")[["intime", "window_end", "row_id"]]
        keep, parts = set(win["stay_id"]), []
        for chunk in pd.read_csv(path, usecols=["stay_id", "itemid", time_col],
                                 parse_dates=[time_col], chunksize=2_000_000,
                                 on_bad_lines="skip"):
            chunk = chunk[chunk["itemid"].isin(itemids) & chunk["stay_id"].isin(keep)]
            if chunk.empty:
                continue
            chunk = chunk.join(bounds, on="stay_id")
            chunk = chunk[(chunk[time_col] >= chunk["intime"])
                          & (chunk[time_col] < chunk["window_end"])]
            parts.append(chunk[["itemid", "row_id"]])
        res = pd.concat(parts).drop_duplicates() if parts else pd.DataFrame(
            columns=["itemid", "row_id"])

    return {int(i): set(b["row_id"]) for i, b in res.groupby("itemid")}


def _urine_totals(path, win, verbose):
    """{row_id: total urine volume (mL)} inside each stay's window."""
    all_items = mc.OUTPUT_URINE_ITEMS + mc.OUTPUT_URINE_NEGATIVE_ITEMS
    ids = ", ".join(str(i) for i in all_items)
    neg_ids = ", ".join(str(i) for i in mc.OUTPUT_URINE_NEGATIVE_ITEMS)
    con = _duckdb()
    if con is not None:
        if verbose:
            print("  [tx] duckdb scan of outputevents ...", flush=True)
        con.register("win", win)
        res = con.execute(f"""
            SELECT w.row_id,
                   sum(CASE WHEN e.itemid IN ({neg_ids}) THEN -e.value
                            ELSE e.value END) AS total
            FROM read_csv_auto('{path}', ignore_errors=true) e
            JOIN win w ON e.stay_id = w.stay_id
            WHERE e.itemid IN ({ids}) AND e.value IS NOT NULL
              AND e.value BETWEEN 0 AND {mc.URINE_EVENT_MAX_ML}
              AND e.charttime >= w.intime AND e.charttime < w.window_end
            GROUP BY w.row_id
        """).fetch_df()
        con.close()
    else:
        bounds = win.set_index("stay_id")[["intime", "window_end", "row_id"]]
        keep, parts = set(win["stay_id"]), []
        for chunk in pd.read_csv(path, usecols=["stay_id", "itemid", "charttime", "value"],
                                 parse_dates=["charttime"], chunksize=2_000_000,
                                 on_bad_lines="skip"):
            chunk = chunk[chunk["itemid"].isin(all_items)
                          & chunk["stay_id"].isin(keep)]
            chunk = chunk[chunk["value"].between(0, mc.URINE_EVENT_MAX_ML)]
            if chunk.empty:
                continue
            chunk = chunk.join(bounds, on="stay_id")
            chunk = chunk[(chunk["charttime"] >= chunk["intime"])
                          & (chunk["charttime"] < chunk["window_end"])]
            chunk["value"] = np.where(
                chunk["itemid"].isin(mc.OUTPUT_URINE_NEGATIVE_ITEMS),
                -chunk["value"], chunk["value"])
            parts.append(chunk[["row_id", "value"]])
        res = (pd.concat(parts).groupby("row_id")["value"].sum().reset_index(name="total")
               if parts else pd.DataFrame(columns=["row_id", "total"]))
    return dict(zip(res["row_id"].astype(int), res["total"].astype(float)))


# ------------------------------------------------------------- derived signal

def _severity(df, blocks):
    """Severity from the observation window only -- no outcome, no full-stay info.

    This exists because `_criticalness` weights the death flag at 1.2 against
    two standardised terms, so it correlates with mortality at AUROC ~0.97. That
    is fine for an auxiliary *training* target, but it makes it useless as an
    evaluation stratum: "top decile of criticalness" is very nearly "the
    patients who died", which is the same degeneracy `high_risk_analysis` was
    fixed to avoid (HANDOVER 6.1).

    A mini-SOFA instead: organ-support exposure plus physiologic derangement,
    all measured inside `[intime, intime + obs_window)`. Every component is
    already a model input, which is fine -- the stratum has to be independent of
    the model's *decisions* and of the *outcome*, not of the features.

    Components are robust-z-scored (median / IQR) so a single wild lab does not
    dominate, oriented so larger is sicker, and averaged over whichever are
    available.
    """
    cols = pd.concat([b for b in blocks.values() if b is not None and b.shape[1]],
                     axis=1) if blocks else pd.DataFrame(index=df.index)

    # (column, +1 if high is sicker, -1 if low is sicker)
    COMPONENTS = [
        ("tx_any_vasopressor", +1), ("tx_invasive_ventilation", +1),
        ("tx_crrt", +1), ("tx_n_interventions", +1),
        ("lab_lactate_max", +1), ("lab_creatinine_max", +1), ("lab_bun_max", +1),
        ("vit_gcs_motor_last", -1), ("vit_gcs_verbal_last", -1),
        ("vit_sbp_min", -1), ("vit_spo2_min", -1), ("vit_resp_rate_max", +1),
    ]

    parts, used = [], []
    for name, sign in COMPONENTS:
        if name not in cols.columns:
            continue
        v = cols[name].to_numpy(dtype=np.float64)
        if not np.isfinite(v).any():
            continue
        med = np.nanmedian(v)
        q1, q3 = np.nanpercentile(v[np.isfinite(v)], [25, 75])
        scale = max(q3 - q1, 1e-6)
        z = sign * (np.where(np.isfinite(v), v, med) - med) / scale
        parts.append(np.clip(z, -4.0, 4.0))
        used.append(name)

    if not parts:
        return np.zeros(len(df), dtype=np.float32), []

    raw = np.mean(parts, axis=0)
    lo, hi = np.percentile(raw, [1, 99])
    score = np.clip((raw - lo) / max(hi - lo, 1e-8), 0.0, 1.0)
    return score.astype(np.float32), used


def _criticalness(df, treatment):
    """Continuous severity proxy used as the auxiliary regression target.

    Follows the SBSCGM multi-task setup (arXiv:2508.00615 sec. 3.4): a
    normalised blend of intervention burden, ICU length of stay, and discharge
    disposition. It is a *label*, derived from the full stay, and is only ever
    read on training nodes inside the loss -- never a model input.

    Note the death flag carries weight 1.2 against two standardised terms, so
    this correlates with mortality at AUROC ~0.97 on the MIMIC cohort. That is
    acceptable for a training target but disqualifies it as an evaluation
    stratum -- use `_severity` for that.
    """
    los = np.log1p(df["los_hours"].to_numpy(dtype=np.float64))
    los = (los - los.mean()) / (los.std() + 1e-8)
    burden = treatment["tx_n_interventions"].to_numpy(dtype=np.float64)
    burden = (burden - burden.mean()) / (burden.std() + 1e-8)
    died = df["label"].to_numpy(dtype=np.float64)

    raw = 0.35 * los + 0.35 * burden + 1.2 * died
    lo, hi = np.percentile(raw, [1, 99])
    return np.clip((raw - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)


# ----------------------------------------------------------------- top level

# Bump when the *block-building* semantics change (itemid sets, aggregates,
# window logic). Expensive: it invalidates the 18 GB and 42 GB scans.
_BLOCK_VERSION = 2
# Bump when *assembly* semantics change (cohort filters, packing, derived
# columns). Cheap: blocks are reused, only the assembled table is rebuilt.
_COHORT_VERSION = 4

_ROW_KEYS = ("source", "task", "obs_window_hours", "min_los_hours",
             "max_patients", "cohort_seed")


def row_signature(cfg):
    """Hash of the fields that decide *which patients* are in the cohort.

    Feature blocks are cached against this rather than against
    `cohort_signature`, so enabling one more block reuses the multi-GB scans
    already done for the others instead of invalidating them.
    """
    d = cfg["data"]
    payload = {k: d.get(k) for k in _ROW_KEYS}
    payload["_v"] = _BLOCK_VERSION
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha1(blob.encode()).hexdigest()[:10]


def cohort_signature(cfg):
    """Hash of only the fields that change the extracted cohort."""
    d = cfg["data"]
    keys = _ROW_KEYS + ("blocks", "require_nonempty_blocks")
    payload = {k: d.get(k) for k in keys}
    payload["_v"] = (_BLOCK_VERSION, _COHORT_VERSION)
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha1(blob.encode()).hexdigest()[:10]


def _block_present(block):
    """Boolean mask: did this patient have *any* measurement in this block?

    Not `notna().any()`. `_aggregate_events` fills the `_count` columns with
    zero on purpose -- a concept that was never measured has count 0, not a
    missing count -- so every row is non-null in those columns and a naive
    null check calls every patient covered. The count columns are precisely
    the ones that answer the question, when they exist.
    """
    counts = [c for c in block.columns if c.endswith("_count")]
    if counts:
        return (block[counts].fillna(0).to_numpy() > 0).any(axis=1)
    return block.notna().any(axis=1).to_numpy()


def _drop_uncovered(df, blocks, required, verbose=True):
    """Drop patients for whom a required feature block is entirely absent.

    Public MIMIC mirrors are not always complete. The Kaggle
    `akshaybe/updated-mimic-iv` copy ships a `chartevents.csv` that is truncated
    mid-record and covers only the first ~38% of ICU stays in `subject_id`
    order, so most patients have no vitals at all.

    Leaving them in is worse than dropping them. `impute_and_scale` appends a
    missingness indicator per column *because* missingness in the ICU is
    informative -- a lab not drawn is a clinical decision. When the missingness
    is instead an artefact of a truncated download, those indicators encode
    "this patient's subject_id sorts below the cut point", and the model learns
    the defect as signal.

    `subject_id` is randomly assigned in MIMIC-IV, so the surviving cohort is
    not clinically selected -- but that is a property of this particular
    truncation, and is asserted rather than assumed by the caller.
    """
    if not required:
        return df, blocks, np.ones(len(df), dtype=bool)

    keep = np.ones(len(df), dtype=bool)
    for name in required:
        block = blocks.get(name)
        if block is None:
            raise ValueError(f"require_nonempty_blocks names {name!r}, which was not built")
        present = _block_present(block)
        if verbose:
            print(f"  [{name}] present for {present.sum():,} / {len(present):,} "
                  f"patients ({100 * present.mean():.1f}%)")
        keep &= present

    if keep.all():
        return df, blocks, keep
    if verbose:
        print(f"  dropped {(~keep).sum():,} patients lacking a required block "
              f"-> {keep.sum():,} remain")
    df = df[keep].reset_index(drop=True)
    blocks = {k: v[keep].reset_index(drop=True) for k, v in blocks.items()}
    return df, blocks, keep


def _cached_block(name, sig, cache_dir, build, force=False, verbose=True):
    """Build one feature block, memoised on disk.

    The `labs` and `vitals` blocks each cost a full scan of a multi-GB event
    table. On a hosted kernel with a wall-clock session limit, losing those to
    an unrelated failure later in the pipeline is expensive, so every block is
    written the moment it is built and reused on the next run.
    """
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"block_{sig}_{name}.parquet")
    if os.path.exists(path) and not force:
        block = pd.read_parquet(path)
        if verbose:
            print(f"  [{name}] cached ({block.shape[1]} cols)")
        return block
    t0 = time.time()
    block = build()
    block.to_parquet(path, index=False)
    if verbose:
        print(f"  [{name}] built {block.shape[1]} cols in {time.time() - t0:.1f}s "
              f"-> {path}")
    return block


def build_cohort(cfg, verbose=True):
    """Extract (or load a cached) MIMIC-IV cohort.

    Returns `(X, y, feature_names, meta)` with the same contract as the mock
    generator. `X` keeps NaN for genuinely unmeasured values -- imputation
    happens later, fit on the training split only.
    """
    d = cfg["data"]
    raw_dir = d.get("raw_dir", "data/raw")
    cache_dir = d.get("cache_dir", "data/processed")
    root = find_root(raw_dir)
    if root is None:
        raise FileNotFoundError(
            f"no MIMIC-IV tree under {raw_dir}. Run:\n"
            f"    python scripts/fetch_mimic.py --tier core --tier events"
        )

    sig = cohort_signature(cfg)
    row_sig = row_signature(cfg)
    cache = os.path.join(cache_dir, f"cohort_{sig}.csv.gz")
    if os.path.exists(cache) and not d.get("force_rebuild", False):
        if verbose:
            print(f"[mimic] cached cohort {cache}")
        return _unpack(pd.read_csv(cache), cfg)

    if verbose:
        print(f"[mimic] building cohort from {root}")
    df, steps = _base_cohort(root, cfg)
    if verbose:
        for name, n in steps:
            print(f"  {name:28s} {n:>8,}")

    max_patients = d.get("max_patients")
    if max_patients and len(df) > max_patients:
        # Subsample *patients*, stratified on the label so prevalence survives.
        rng = np.random.default_rng(d.get("cohort_seed", cfg["seed"]))
        keep = np.concatenate([
            rng.permutation(np.flatnonzero(df["label"].to_numpy() == cls))[
                : int(round(max_patients * (df["label"] == cls).mean()))
            ]
            for cls in (0, 1)
        ])
        df = df.iloc[np.sort(keep)].reset_index(drop=True)
        if verbose:
            print(f"  {'subsampled':28s} {len(df):>8,}")

    enabled = set(d.get("blocks") or
                  ["demographics", "comorbidity", "labs", "vitals", "treatment"])
    builders = {
        "demographics": lambda: _demographics_block(df),
        "comorbidity": lambda: _comorbidity_block(root, df),
        "labs": lambda: _aggregate_events(
            root, df, "hosp/labevents.csv", "hadm_id", "charttime", "valuenum",
            mc.LAB_ITEMS, "lab", verbose=verbose),
        "vitals": lambda: _aggregate_events(
            root, df, "icu/chartevents.csv", "stay_id", "charttime", "valuenum",
            mc.CHART_ITEMS, "vit", verbose=verbose),
        "treatment": lambda: _treatment_block(root, df, verbose=verbose),
    }
    blocks, views = {}, {}
    for name in ("demographics", "comorbidity", "labs", "vitals", "treatment"):
        if name in enabled:
            blocks[name] = _cached_block(name, row_sig, cache_dir, builders[name],
                                         force=d.get("force_rebuild", False),
                                         verbose=verbose)

    # Drop blocks that came back entirely empty (source table not downloaded).
    blocks = {k: v for k, v in blocks.items()
              if v.shape[1] and not v.isna().all().all()}

    # Then drop patients for whom a required block is missing entirely, which
    # is how a partial mirror shows up (see `_drop_uncovered`).
    df, blocks, _kept = _drop_uncovered(
        df, blocks, [b for b in (d.get("require_nonempty_blocks") or []) if b in blocks],
        verbose=verbose)

    cursor = 0
    for name, block in blocks.items():
        views[name] = list(range(cursor, cursor + block.shape[1]))
        cursor += block.shape[1]

    features = pd.concat(list(blocks.values()), axis=1)
    crit = _criticalness(df, blocks.get("treatment",
                                        pd.DataFrame({"tx_n_interventions": np.zeros(len(df))})))
    sev, sev_used = _severity(df, blocks)
    if verbose:
        print(f"  [severity] {len(sev_used)} components: {', '.join(sev_used)}")

    packed = features.copy()
    packed.insert(0, "subject_id", df["subject_id"].to_numpy())
    packed.insert(1, "hadm_id", df["hadm_id"].to_numpy())
    packed.insert(2, "stay_id", df["stay_id"].to_numpy())
    packed.insert(3, "label", df["label"].to_numpy())
    packed.insert(4, "criticalness", crit)
    packed.insert(5, "los_hours", df["los_hours"].to_numpy())
    packed.insert(6, "severity", sev)

    os.makedirs(cache_dir, exist_ok=True)
    packed.to_csv(cache, index=False)
    with open(os.path.join(cache_dir, f"cohort_{sig}.views.json"), "w") as f:
        json.dump({"views": views, "steps": steps}, f, indent=2, default=str)
    if verbose:
        print(f"[mimic] wrote {cache}  ({len(packed):,} patients, "
              f"{features.shape[1]} features, prevalence "
              f"{packed['label'].mean():.3f})")
    return _unpack(packed, cfg)


_ID_COLS = ("subject_id", "hadm_id", "stay_id", "label", "criticalness",
            "los_hours", "severity")


def _unpack(packed, cfg):
    """Split the cached table into the (X, y, names, meta) contract."""
    sig = cohort_signature(cfg)
    view_path = os.path.join(cfg["data"].get("cache_dir", "data/processed"),
                             f"cohort_{sig}.views.json")
    views = {}
    if os.path.exists(view_path):
        with open(view_path) as f:
            views = {k: list(map(int, v)) for k, v in json.load(f)["views"].items()}

    names = [c for c in packed.columns if c not in _ID_COLS]
    X = packed[names].to_numpy(dtype=np.float32)
    y = packed["label"].to_numpy(dtype=np.int64)

    # Columns that are already 0/1 drive the Jaccard half of the hybrid
    # similarity (arXiv:2508.00615 sec. 3.1); everything else is continuous.
    finite = np.where(np.isfinite(X), X, 0.0)
    binary = np.all((finite == 0.0) | (finite == 1.0), axis=0)

    meta = {
        "subject_id": packed["subject_id"].to_numpy(),
        "hadm_id": packed["hadm_id"].to_numpy(),
        "stay_id": packed["stay_id"].to_numpy(),
        "criticalness": packed["criticalness"].to_numpy(dtype=np.float32),
        "severity": packed["severity"].to_numpy(dtype=np.float32),
        # Time axis for the concordance index. ICU length of stay with
        # in-hospital death as the event -- survivors censored at discharge.
        "los_hours": packed["los_hours"].to_numpy(dtype=np.float32),
        "views": views,
        "binary_cols": np.flatnonzero(binary),
    }
    if len(y) and y.sum() == 0:
        warnings.warn("cohort has no positive labels -- check the window filters")
    return X, y, names, meta

Writing /kaggle/working/src/mimic_iv.py


In [4]:
%%writefile /kaggle/working/src/instrument.py
"""Direct-instrumentation counters, shared across every Architecture 4R module.

Why this exists
----------------
The project's own history has a documented case of a compute-saving claim
that was never actually measured -- Architecture 1's efficiency figure was
exit-depth arithmetic, not a count of work performed (`old/ARCHITECTURES.md`
SS3, "measured rather than asserted"). Architecture 2 and 3 fixed this by
counting real operations; `ppr_push.py`'s `ledger` dict is one such counter,
built ad hoc per module. This module is the single shared counting primitive
so `local_push.py` and the Phase-1 driver can't drift into two different
definitions of "an edge was traversed."
"""

import time
from contextlib import contextmanager
from dataclasses import dataclass, field


@dataclass
class OpCounter:
    edges: int = 0
    nodes: int = 0
    queue_pops: int = 0
    madds: int = 0
    phase_s: dict = field(default_factory=dict)
    peak_mem_bytes: int = 0

    def count_edge(self, n: int = 1) -> None:
        self.edges += n

    def count_node_visit(self, n: int = 1) -> None:
        self.nodes += n

    def count_queue_pop(self, n: int = 1) -> None:
        self.queue_pops += n

    def count_madd(self, n: int = 1) -> None:
        self.madds += n

    @contextmanager
    def phase(self, name: str):
        """Wall-clock (and, if torch+CUDA is available, peak-memory) for one phase.

        Accumulates into `phase_s[name]` across repeated calls with the same
        name, so a phase entered multiple times per query (e.g. "diffusion"
        across several checkpoints) reports total time, not just the last call.
        """
        peak_before = self._cuda_peak_bytes()
        t0 = time.perf_counter()
        try:
            yield
        finally:
            elapsed = time.perf_counter() - t0
            self.phase_s[name] = self.phase_s.get(name, 0.0) + elapsed
            peak_after = self._cuda_peak_bytes()
            if peak_after is not None:
                self.peak_mem_bytes = max(self.peak_mem_bytes, peak_after)
            del peak_before

    @staticmethod
    def _cuda_peak_bytes():
        try:
            import torch
            if torch.cuda.is_available():
                return torch.cuda.max_memory_allocated()
        except Exception:
            pass
        return None

    def snapshot(self) -> dict:
        return {
            "edges": self.edges,
            "nodes": self.nodes,
            "queue_pops": self.queue_pops,
            "madds": self.madds,
            "phase_s": dict(self.phase_s),
            "peak_mem_bytes": self.peak_mem_bytes,
        }

    def merge(self, other: "OpCounter") -> None:
        """Fold another counter's totals into this one (e.g. per-query -> per-batch)."""
        self.edges += other.edges
        self.nodes += other.nodes
        self.queue_pops += other.queue_pops
        self.madds += other.madds
        for name, secs in other.phase_s.items():
            self.phase_s[name] = self.phase_s.get(name, 0.0) + secs
        self.peak_mem_bytes = max(self.peak_mem_bytes, other.peak_mem_bytes)

Writing /kaggle/working/src/instrument.py


In [5]:
%%writefile /kaggle/working/src/encoder_mv.py
"""Architecture 4R's weak multi-view encoder.

Why "weak" is load-bearing, not a hedge
---------------------------------------
Architecture 2 (`ppr_push.py`) already showed that local-push diffusion adds
real signal (+0.0207 AUROC) *on top of a shallow, graph-free encoder* -- and
adds nothing on top of an already-strong tabular score (the closed graph-
residual experiment, `old/ARCHITECTURES.md` SS2/SS3). So this encoder is
deliberately capped in capacity: no boosted-tree score, no leaf identifier,
no strong teacher logit, and no label-derived feature is allowed anywhere
near it. Making it stronger would put diffusion back in the regime where the
project has already measured that it buys nothing.

Five views, one encoder each: demographics, comorbidity, labs, vitals,
treatment -- the same five blocks `src/tabular.py`'s `STAGE_BLOCKS` groups
into stages, kept separate here because Architecture 4R's graph needs one
similarity structure per view, not one per cascade stage.
"""

from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

VIEW_OUT_DIMS = {
    "demographics": 16, "comorbidity": 32, "labs": 32,
    "vitals": 32, "treatment": 32,
}
VIEW_ORDER = tuple(VIEW_OUT_DIMS)  # fixed iteration order everywhere


# --------------------------------------------------------------- preprocessing

@dataclass
class _ViewStats:
    median: np.ndarray
    lo: np.ndarray
    hi: np.ndarray
    scale: np.ndarray


class ViewPreprocessors:
    """Per-view median/winsorize/scale, fit train-only -- one `_ViewStats` per view.

    Mirrors `src.data_prep.impute_and_scale`'s train-only discipline, but
    per-view and with winsorization (a self-tuned graph is more sensitive to
    a single extreme value than a tree ensemble is, so clipping matters more
    here than it does for `data_prep`'s tabular path).
    """

    def __init__(self, stats: dict[str, _ViewStats]):
        self.stats = stats

    def transform(self, view_raw: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
        out = {}
        for name, x in view_raw.items():
            s = self.stats[name]
            x = np.asarray(x, dtype=np.float64)
            x = np.where(np.isnan(x), s.median, x)
            x = np.clip(x, s.lo, s.hi)
            out[name] = ((x - s.median) / s.scale).astype(np.float32)
        return out


def fit_view_preprocessors(view_raw: dict[str, np.ndarray], train_mask) -> ViewPreprocessors:
    """Fit medians/winsorization limits/scale on training rows only per view."""
    train_mask = np.asarray(train_mask, dtype=bool)
    stats = {}
    for name, x in view_raw.items():
        x = np.asarray(x, dtype=np.float64)
        xt = x[train_mask]
        median = np.nanmedian(xt, axis=0)
        median = np.where(np.isnan(median), 0.0, median)
        q1 = np.nanpercentile(xt, 25, axis=0)
        q3 = np.nanpercentile(xt, 75, axis=0)
        iqr = np.where(np.isnan(q3 - q1) | (q3 - q1 < 1e-6), 1.0, q3 - q1)
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        scale = np.where(iqr < 1e-6, 1.0, iqr)
        stats[name] = _ViewStats(median=median, lo=lo, hi=hi, scale=scale)
    return ViewPreprocessors(stats)


def observed_fraction(view_raw: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Per-patient, per-view observed-feature fraction r_i^(v) -- computed on
    the *raw* (pre-imputation) arrays, since after `ViewPreprocessors.transform`
    every NaN has already been filled in and this signal would be lost."""
    out = {}
    for name, x in view_raw.items():
        x = np.asarray(x, dtype=np.float64)
        out[name] = 1.0 - np.isnan(x).mean(axis=1)
    return out


# -------------------------------------------------------------------- encoder

class ViewEncoder(nn.Module):
    """z^(v) = LN(W2 GELU(W1 x + b1) + b2), one hidden layer, width 64."""

    def __init__(self, in_dim: int, out_dim: int, hidden: int = 64, dropout: float = 0.15):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
        self.drop = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.drop(F.gelu(self.fc1(x)))
        return self.ln(self.fc2(h))


class MultiViewEncoder(nn.Module):
    """Five per-view encoders + one residual fusion layer -> h_i^(0).

    A view that is entirely missing for a patient (`reliability_raw[v][i] ==
    0`) has its encoded code replaced by a learned, view-specific missing
    embedding -- so a patient with a whole block absent gets a deterministic,
    trainable representation instead of an encoder output computed from
    imputed medians pretending to be data.
    """

    def __init__(self, in_dims: dict[str, int], fusion_dim: int = 64,
                 dropout: float = 0.15, out_dims: dict[str, int] | None = None):
        super().__init__()
        out_dims = out_dims or VIEW_OUT_DIMS
        self.view_names = [v for v in VIEW_ORDER if v in in_dims]
        missing = [v for v in in_dims if v not in VIEW_OUT_DIMS]
        if missing:
            raise ValueError(f"unknown view name(s), not in VIEW_OUT_DIMS: {missing}")

        self.encoders = nn.ModuleDict({
            v: ViewEncoder(in_dims[v], out_dims[v], dropout=dropout)
            for v in self.view_names
        })
        self.missing_embed = nn.ParameterDict({
            v: nn.Parameter(torch.zeros(out_dims[v])) for v in self.view_names
        })
        concat_dim = sum(out_dims[v] for v in self.view_names)
        self.fusion_skip = nn.Linear(concat_dim, fusion_dim)
        self.fusion_mlp = nn.Sequential(
            nn.Linear(concat_dim, fusion_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
        )
        self.fusion_ln = nn.LayerNorm(fusion_dim)
        self.fusion_dim = fusion_dim

    def forward(self, view_x: dict[str, torch.Tensor],
                view_observed_frac: dict[str, torch.Tensor]) -> dict:
        view_z, reliability_raw = {}, {}
        for v in self.view_names:
            r = view_observed_frac[v]  # [n], in [0, 1]
            z = self.encoders[v](view_x[v])
            missing_row = (r <= 0.0)
            if missing_row.any():
                z = torch.where(
                    missing_row.unsqueeze(1),
                    self.missing_embed[v].unsqueeze(0).expand(z.size(0), -1),
                    z,
                )
            view_z[v] = z
            reliability_raw[v] = r

        concat = torch.cat([view_z[v] for v in self.view_names], dim=1)
        h0 = self.fusion_ln(self.fusion_skip(concat) + self.fusion_mlp(concat))
        return {"h0": h0, "view_z": view_z, "reliability_raw": reliability_raw}

    def n_trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

Writing /kaggle/working/src/encoder_mv.py


In [6]:
%%writefile /kaggle/working/src/graph_density.py
"""Architecture 4R's density-adaptive, per-view, inductive patient graph.

Why this is a new module rather than an extension of `src/graph_build.py`
--------------------------------------------------------------------------
`graph_build.py` is fully transductive (every split's rows sit in the graph
from construction time) and uses one shared `k`/similarity metric across
every view. Neither property is safe to reuse here: Architecture 4R's own
methodology requires held-out patients to attach only to a *frozen* training
reference graph (no held-out-to-held-out edge may ever exist), and requires
a distinct construction per view (self-tuning local scale, density-adaptive
neighbourhood size, a per-view qualification threshold). `graph_build.py`'s
`_topk_numpy`/`_topk_torch` scoring kernels are still fine primitives for the
raw distance computation and are used here for that purpose only.

Explicit design decision (open question in the Phase 0/1 implementation
plan): the per-view qualification threshold `T_v` is always fit under the
**adaptive** rule (each training node's own adaptive k_i), even when this
module is asked to build the fixed-k=16 ablation arm. `T_v` is a property of
the view's own distance geometry, not of which k a given experimental arm
happens to use, and sharing it keeps the fixed-k vs. adaptive-k comparison
apples-to-apples (only k differs; qualification, reciprocity radius, and
novelty ECDF are all identical between the two arms by construction).

Topology only, no encoder here
-------------------------------
Everything in this module (candidate search, sigma/rho, adaptive k,
qualification, reciprocity, novelty, cold start) depends only on
*preprocessed features* -- never on the encoder's learned codes `z`. The
encoder is trained jointly with the edge scorer, so `z` changes every
optimizer step; baking it into a frozen topology would either go stale
immediately or force a topology rebuild every step, which the training
procedure explicitly forbids (at most one rebuild, after epoch 10). The
learned, `z`-dependent edge scoring (GATv2-style scorer + view-reliability
fusion) lives at the bottom of this module and is meant to be called fresh
every forward pass against this module's frozen topology.
"""

from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors

VIEW_METRIC = {
    "demographics": "euclidean", "comorbidity": "cosine", "labs": "euclidean",
    "vitals": "euclidean", "treatment": "cosine",
}


@dataclass
class ViewGraphConfig:
    k_cand: int = 64
    k0: int = 16
    gamma: float = 0.5              # in {0.25, 0.5, 0.75}, selected on val
    k_min_adaptive: int = 4
    k_max_adaptive: int = 48
    percentile_qualify: float = 99.5
    fixed_k: int | None = None      # None -> adaptive; int (16) -> fixed-k ablation arm


# ------------------------------------------------------------- candidate search

def candidate_search(X_train: np.ndarray, X_query: np.ndarray, k_cand: int,
                      metric: str):
    """Exact k_cand-nearest-neighbour search, `X_query` against `X_train`.

    Returns (idx [n_query, k_cand], dist [n_query, k_cand]), sorted ascending.
    Exact by default -- brute-force is well within reach at this cohort size
    (~17k patients per view); `audit_ann_recall` gates any future ANN use.
    """
    nn_ = NearestNeighbors(n_neighbors=min(k_cand, len(X_train)), metric=metric,
                           algorithm="brute")
    nn_.fit(X_train)
    dist, idx = nn_.kneighbors(X_query)
    return idx, dist


def audit_ann_recall(X_train: np.ndarray, k_cand: int, metric: str, ann_fn) -> dict:
    """Compare an approximate search function against exact search.

    `ann_fn(X_train, X_train, k_cand, metric) -> (idx, dist)`, same contract
    as `candidate_search`. Returns recall@k_cand statistics; the caller (not
    this function) decides whether to actually use the ANN path based on the
    thresholds in the Architecture 4R spec (mean >= 0.98, min-decile >= 0.95).
    """
    exact_idx, _ = candidate_search(X_train, X_train, k_cand, metric)
    ann_idx, _ = ann_fn(X_train, X_train, k_cand, metric)
    n = exact_idx.shape[0]
    recall = np.array([
        len(set(exact_idx[i]) & set(ann_idx[i])) / k_cand for i in range(n)
    ])
    deciles = np.quantile(recall, np.linspace(0.0, 0.9, 10))
    return {
        "recall_at_k_mean": float(recall.mean()),
        "recall_at_k_min_decile": float(deciles.min()),
        "used_ann": False,  # the caller sets this True only after checking thresholds
    }


# ------------------------------------------------------- density / adaptive k

def self_tuning_sigma(dist_to_k0: np.ndarray) -> np.ndarray:
    """sigma_i = median distance to the nearest k0 candidates."""
    return np.median(dist_to_k0, axis=1)


def local_density(dist_candidates: np.ndarray, sigma_query: np.ndarray,
                   sigma_candidates: np.ndarray) -> np.ndarray:
    """rho_i^(v), self-tuning kernel over the full k_cand candidate set.

    `sigma_candidates` is [n_query, k_cand] -- sigma_j for each of node i's
    candidates j, gathered by the caller from the training set's own sigma.
    """
    denom = 2.0 * sigma_query[:, None] * sigma_candidates + 1e-8
    return np.exp(-(dist_candidates ** 2) / denom).sum(axis=1)


def standardize_log_density(rho: np.ndarray, train_median: float, train_mad: float) -> np.ndarray:
    log_rho = np.log(rho + 1e-8)
    denom = train_mad if train_mad > 1e-8 else 1.0
    return (log_rho - train_median) / denom


def adaptive_k(q: np.ndarray, k0: int, gamma: float, k_min: int, k_max: int) -> np.ndarray:
    k = np.round(k0 * np.exp(gamma * q)).astype(int)
    return np.clip(k, k_min, k_max)


def delta(dist: np.ndarray, sigma_i, sigma_j) -> np.ndarray:
    """Dimensionless self-tuned distance Delta_ij = d^2 / (2 sigma_i sigma_j + eps)."""
    return (dist ** 2) / (2.0 * sigma_i * sigma_j + 1e-8)


def fit_qualification_threshold(train_deltas_loo: np.ndarray, percentile: float) -> float:
    return float(np.percentile(train_deltas_loo, percentile))


def base_edge_weight(delta_ij: np.ndarray, reciprocal_mask: np.ndarray) -> np.ndarray:
    return np.exp(-delta_ij) * (0.5 + 0.5 * reciprocal_mask.astype(np.float64))


def graph_availability(qualified_ref_count: np.ndarray, reliable_view_count: np.ndarray,
                        k_min: int = 4) -> np.ndarray:
    return (qualified_ref_count >= k_min) & (reliable_view_count >= 1)


# --------------------------------------------------------- per-view reference

@dataclass
class _ViewReference:
    X_train: np.ndarray
    metric: str
    sigma_train: np.ndarray          # [n_train], leave-one-out
    k_train: np.ndarray              # [n_train], adaptive k actually used to build edges
    reciprocity_radius: np.ndarray   # [n_train], distance to the k_train-th neighbour
    T_v: float                       # qualification threshold, fit under the adaptive rule
    sigma_log_median: float
    sigma_log_mad: float             # (of standardized log-density, for reuse if needed)
    ecdf_values: np.ndarray          # sorted leave-one-out l_i^(v), for novelty percentile
    edge_index_ref: np.ndarray       # [2, E] train-train edges (row 0: src i, row 1: dst j)
    edge_weight_ref: np.ndarray      # [E] base weight b_ij^(v)
    r_min: float                     # reliability threshold for this view


class ReferenceGraph:
    """Frozen, train-only topology for every view, plus everything needed to
    inductively attach held-out patients without ever touching it."""

    def __init__(self, views: dict[str, _ViewReference], cfg: ViewGraphConfig,
                 train_qualified_count: np.ndarray | None = None,
                 train_reliable_view_count: np.ndarray | None = None):
        self.views = views
        self.cfg = cfg
        # Per-train-node coverage bookkeeping, same rule `attach()` applies to
        # held-out queries (`graph_availability`) -- needed because Arch4R's
        # training is transductive (every train node is also scored as its
        # own query against this same frozen graph), so a train node can be
        # just as "cold start" (too few qualifying same-view neighbours) as
        # a held-out one, and the fallback path must trigger for both the
        # same way. Optional/None only for backward compatibility with any
        # caller constructing a bare `ReferenceGraph` directly in a test.
        self.train_qualified_count = train_qualified_count
        self.train_reliable_view_count = train_reliable_view_count

    def train_graph_available(self, k_min: int | None = None) -> np.ndarray:
        """Per-train-node availability, computed the identical way `attach()`
        computes it for held-out queries -- see `graph_availability`."""
        if self.train_qualified_count is None or self.train_reliable_view_count is None:
            raise ValueError("ReferenceGraph was not built with train-side coverage "
                             "counts (use build_reference_graph, not a bare constructor)")
        k_min = self.cfg.k_min_adaptive if k_min is None else k_min
        return graph_availability(self.train_qualified_count, self.train_reliable_view_count,
                                  k_min=k_min)

    # ---------------------------------------------------------------- attach
    def attach(self, query_views: dict[str, np.ndarray],
               query_reliability: dict[str, np.ndarray]) -> "AttachedGraph":
        """Attach a batch of held-out queries as new source nodes.

        Never mutates `self.views` (no reference-reference edge, density
        statistic, or fitted parameter can move because of held-out data),
        and never connects one query to another -- only query -> train edges.
        """
        n_query = next(iter(query_views.values())).shape[0]
        per_view_edges = {}
        novelty_per_view = np.zeros((n_query, len(self.views)))
        reliable_mask = np.zeros((n_query, len(self.views)), dtype=bool)
        qualified_count = np.zeros(n_query, dtype=int)

        for vi, (name, ref) in enumerate(self.views.items()):
            Xq = query_views[name]
            k_cand = min(self.cfg.k_cand, len(ref.X_train))
            idx, dist = candidate_search(ref.X_train, Xq, k_cand, ref.metric)

            k0 = min(self.cfg.k0, k_cand)
            sigma_q = self_tuning_sigma(dist[:, :k0])
            sigma_cand = ref.sigma_train[idx]
            delta_ij = delta(dist, sigma_q[:, None], sigma_cand)

            # novelty: median Delta over the same k0 neighbours used for sigma_q
            l_q = np.median(delta_ij[:, :k0], axis=1)
            novelty_per_view[:, vi] = np.searchsorted(ref.ecdf_values, l_q) / max(
                len(ref.ecdf_values), 1)

            r_q = query_reliability[name]
            reliable = r_q >= ref.r_min
            reliable_mask[:, vi] = reliable

            k_use = (np.full(n_query, self.cfg.fixed_k) if self.cfg.fixed_k is not None
                    else None)  # adaptive k computed per-query below if needed
            if k_use is None:
                # Adaptive k for a query uses the *query's own* density, consistent
                # with how a training node's k_i was derived from its own rho_i.
                rho_q = local_density(dist, sigma_q, sigma_cand)
                q_std = standardize_log_density(rho_q, ref.sigma_log_median, ref.sigma_log_mad)
                k_use = adaptive_k(q_std, self.cfg.k0, self.cfg.gamma,
                                   self.cfg.k_min_adaptive, self.cfg.k_max_adaptive)
            k_use = np.minimum(k_use, k_cand)

            qualifies = delta_ij <= ref.T_v
            keep = np.zeros_like(qualifies)
            for r in range(n_query):
                keep[r, :k_use[r]] = qualifies[r, :k_use[r]]

            recip_radius = ref.reciprocity_radius[idx]
            reciprocal = dist <= recip_radius
            w = base_edge_weight(delta_ij, reciprocal)

            src = np.repeat(np.arange(n_query), k_cand)[keep.reshape(-1)]
            dst = idx.reshape(-1)[keep.reshape(-1)]
            wt = w.reshape(-1)[keep.reshape(-1)]
            per_view_edges[name] = (src, dst, wt)
            qualified_count += keep.sum(axis=1)

        novelty = np.zeros(n_query)
        for r in range(n_query):
            rel = reliable_mask[r]
            if not rel.any():
                novelty[r] = 1.0
                continue
            weights = np.array(list(query_reliability[name][r] for name in self.views))
            weights = weights * rel
            weights = weights / (weights.sum() + 1e-8)
            novelty[r] = float((weights * novelty_per_view[r]).sum())

        n_reliable_views = reliable_mask.sum(axis=1)
        available = graph_availability(qualified_count, n_reliable_views,
                                       k_min=self.cfg.k_min_adaptive)

        return AttachedGraph(
            per_view_edges=per_view_edges, novelty=novelty,
            graph_available=available, n_query=n_query,
        )


@dataclass
class AttachedGraph:
    """Per-view (query -> train) edges for one batch of held-out or training queries."""
    per_view_edges: dict[str, tuple]   # name -> (src[E], dst[E], base_weight[E])
    novelty: np.ndarray                # [n_query], in [0, 1]
    graph_available: np.ndarray        # [n_query], bool
    n_query: int


# ------------------------------------------------------------ building it all

def build_reference_graph(train_views: dict[str, np.ndarray],
                          train_reliability: dict[str, np.ndarray],
                          cfg: ViewGraphConfig, metrics: dict[str, str] | None = None,
                          seed: int = 42) -> ReferenceGraph:
    """Build the frozen train-only reference topology for every view.

    `train_views[name]` is that view's preprocessed (imputed/winsorized/
    scaled) feature matrix for training patients only. `train_reliability
    [name]` is the per-patient observed-feature fraction for that view
    (`src.encoder_mv.observed_fraction`, computed on the *raw*, pre-
    imputation array).
    """
    metrics = metrics or VIEW_METRIC
    views = {}
    n_train_total = next(iter(train_views.values())).shape[0]
    qualified_count = np.zeros(n_train_total, dtype=int)
    reliable_view_count = np.zeros(n_train_total, dtype=int)
    for name, X in train_views.items():
        metric = metrics.get(name, "euclidean")
        n = X.shape[0]
        k_cand = min(cfg.k_cand, n - 1)
        k0 = min(cfg.k0, k_cand)

        idx, dist = candidate_search(X, X, k_cand + 1, metric)
        idx, dist = _drop_self(idx, dist, np.arange(n))
        idx, dist = idx[:, :k_cand], dist[:, :k_cand]

        sigma = self_tuning_sigma(dist[:, :k0])
        sigma_cand = sigma[idx]
        rho = local_density(dist, sigma, sigma_cand)
        log_rho = np.log(rho + 1e-8)
        med, mad = float(np.median(log_rho)), float(_mad(log_rho))
        q_std = standardize_log_density(rho, med, mad)
        k_i = adaptive_k(q_std, cfg.k0, cfg.gamma, cfg.k_min_adaptive, cfg.k_max_adaptive)
        k_i = np.minimum(k_i, k_cand)

        delta_ij = delta(dist, sigma[:, None], sigma_cand)

        # T_v under the adaptive rule, always -- see module docstring.
        loo_deltas = np.concatenate([delta_ij[i, :k_i[i]] for i in range(n) if k_i[i] > 0])
        T_v = fit_qualification_threshold(loo_deltas, cfg.percentile_qualify)

        k_use = np.full(n, cfg.fixed_k) if cfg.fixed_k is not None else k_i
        k_use = np.minimum(k_use, k_cand)

        qualifies = delta_ij <= T_v
        keep = np.zeros_like(qualifies)
        for i in range(n):
            keep[i, :k_use[i]] = qualifies[i, :k_use[i]]

        # reciprocity radius: each node's own k_i-th qualified-candidate distance,
        # used both to compute train-train reciprocity here and to let a future
        # held-out query check reciprocity without rebuilding this graph.
        recip_radius = np.array([
            dist[i, k_i[i] - 1] if k_i[i] > 0 else 0.0 for i in range(n)
        ])
        reciprocal = dist <= recip_radius[idx]
        w = base_edge_weight(delta_ij, reciprocal)

        src = np.repeat(np.arange(n), k_cand)[keep.reshape(-1)]
        dst = idx.reshape(-1)[keep.reshape(-1)]
        wt = w.reshape(-1)[keep.reshape(-1)]

        l_i = np.median(delta_ij[:, :k0], axis=1)
        ecdf_values = np.sort(l_i)

        r = train_reliability[name]
        r_min = float(np.clip(np.percentile(r, 5), 0.20, 0.80))

        qualified_count += keep.sum(axis=1)
        reliable_view_count += (r >= r_min).astype(int)

        views[name] = _ViewReference(
            X_train=X, metric=metric, sigma_train=sigma, k_train=k_i,
            reciprocity_radius=recip_radius, T_v=T_v,
            sigma_log_median=med, sigma_log_mad=mad, ecdf_values=ecdf_values,
            edge_index_ref=np.stack([src, dst]), edge_weight_ref=wt, r_min=r_min,
        )
    return ReferenceGraph(views, cfg, train_qualified_count=qualified_count,
                          train_reliable_view_count=reliable_view_count)


def _drop_self(idx: np.ndarray, dist: np.ndarray, self_indices: np.ndarray):
    """Remove each row's self-match (present when querying a set against itself)."""
    n, k = idx.shape
    out_idx = np.empty((n, k - 1), dtype=idx.dtype)
    out_dist = np.empty((n, k - 1), dtype=dist.dtype)
    for i in range(n):
        mask = idx[i] != self_indices[i]
        # If self appears more than once isn't possible for a metric NN search,
        # but guard anyway by taking the first k-1 non-self entries.
        kept = np.flatnonzero(mask)[: k - 1]
        out_idx[i] = idx[i, kept]
        out_dist[i] = dist[i, kept]
    return out_idx, out_dist


def _mad(x: np.ndarray) -> float:
    med = np.median(x)
    return float(np.median(np.abs(x - med)))


def novelty_score(*args, **kwargs):
    """Kept as a thin, direct entry point mirroring the spec's function name;
    the actual computation happens inline in `ReferenceGraph.attach` and
    `build_reference_graph` (train-side), since it needs per-view state that
    only those two call sites hold. See `_ViewReference.ecdf_values`."""
    raise NotImplementedError(
        "use ReferenceGraph.attach(...).novelty (held-out) or "
        "the leave-one-out l_i computed inside build_reference_graph (train)"
    )


# ------------------------------------------------------ dynamic edge scoring

class EdgeScorerGATv2(nn.Module):
    """e_ij^(v) = a_v^T LeakyReLU(W_v[z_i||z_j||abs(z_i-z_j)] + b_v), hidden width 16.

    GATv2-style: the nonlinearity sits *before* the collapse to a scalar via
    `a_v`, which is what actually breaks the "static attention" degeneracy
    (unlike collapsing to a scalar first and applying LeakyReLU after) -- see
    the Architecture 4R review history for the derivation.
    """

    def __init__(self, z_dim: int, hidden: int = 16):
        super().__init__()
        self.proj = nn.Linear(3 * z_dim, hidden)
        self.a = nn.Linear(hidden, 1, bias=False)

    def forward(self, z_i: torch.Tensor, z_j: torch.Tensor) -> torch.Tensor:
        feat = torch.cat([z_i, z_j, (z_i - z_j).abs()], dim=-1)
        h = F.leaky_relu(self.proj(feat))
        e = self.a(h).squeeze(-1)
        return 3.0 * torch.tanh(e / 3.0)  # bounded before exponentiation


class ViewReliability(nn.Module):
    """pi_i^(v) = softmax_v(c_v^T z_i^(v) + beta_v*log(r_i^(v)+eps) + missing_mask)."""

    def __init__(self, z_dims: dict[str, int]):
        super().__init__()
        self.view_names = list(z_dims)
        self.c = nn.ModuleDict({v: nn.Linear(z_dims[v], 1, bias=False) for v in self.view_names})
        self.beta = nn.ParameterDict({v: nn.Parameter(torch.zeros(1)) for v in self.view_names})

    def forward(self, z_view: dict[str, torch.Tensor], reliability_raw: dict[str, torch.Tensor],
                missing_mask: dict[str, torch.Tensor] | None = None) -> dict[str, torch.Tensor]:
        scores = []
        for v in self.view_names:
            s = self.c[v](z_view[v]).squeeze(-1) + self.beta[v] * torch.log(
                reliability_raw[v] + 1e-4)
            if missing_mask is not None and v in missing_mask:
                s = s + torch.where(missing_mask[v], torch.full_like(s, -1e4), torch.zeros_like(s))
            scores.append(s)
        stacked = torch.stack(scores, dim=1)  # [n, n_views]
        pi = torch.softmax(stacked, dim=1)
        return {v: pi[:, i] for i, v in enumerate(self.view_names)}


def fuse_views(per_view_edges: dict[str, tuple], reliability: dict[str, torch.Tensor],
              scorer_output: dict[str, torch.Tensor], n_nodes: int,
              top_k_keep: int = 64):
    """Combine per-view (src, dst, base_weight) edges into one fused, sparse
    graph: A_ij = sum_v sqrt(pi_i^(v) pi_j^(v)) b_ij^(v) exp(e_ij^(v)).

    `reliability[v]` is [n_nodes] (pi^(v) for every node in this batch's
    node set, src and dst share the same indexing). `scorer_output[v]` is
    [E_v], the bounded GATv2 score already computed per view's edges.
    Returns (edge_index [2, E<=n_nodes*top_k_keep], edge_weight [E]).
    """
    all_src, all_dst, all_w = [], [], []
    for v, (src, dst, base_w) in per_view_edges.items():
        if len(src) == 0:
            continue
        src_t = torch.as_tensor(src, dtype=torch.long, device=reliability[v].device)
        dst_t = torch.as_tensor(dst, dtype=torch.long, device=reliability[v].device)
        base_w_t = torch.as_tensor(base_w, dtype=torch.float32, device=reliability[v].device)
        pi_src, pi_dst = reliability[v][src_t], reliability[v][dst_t]
        w = torch.sqrt(pi_src * pi_dst + 1e-12) * base_w_t * torch.exp(scorer_output[v])
        all_src.append(src_t); all_dst.append(dst_t); all_w.append(w)

    if not all_src:
        dev = next(iter(reliability.values())).device
        empty = torch.zeros((2, 0), dtype=torch.long, device=dev)
        return empty, torch.zeros(0, device=dev)

    src = torch.cat(all_src); dst = torch.cat(all_dst); w = torch.cat(all_w)
    # Sum duplicate (src, dst) pairs across views before top-k keep.
    key = src * n_nodes + dst
    uniq_key, inverse = torch.unique(key, return_inverse=True)
    w_summed = torch.zeros(len(uniq_key), dtype=w.dtype, device=w.device)
    w_summed.index_add_(0, inverse, w)
    src_u = (uniq_key // n_nodes)
    dst_u = (uniq_key % n_nodes)

    edge_index, edge_weight = _top_k_per_src(src_u, dst_u, w_summed, top_k_keep)
    return edge_index, edge_weight


def _top_k_per_src(src: torch.Tensor, dst: torch.Tensor, w: torch.Tensor, k: int):
    """Keep at most the top-`k` highest-weight outgoing edges per source node."""
    w_np_detached = w.detach().cpu().numpy()
    perm = np.lexsort((-w_np_detached, src.detach().cpu().numpy()))  # sort by (src, -w)
    src_np, dst_np, w_np = src.cpu().numpy()[perm], dst.cpu().numpy()[perm], w[perm]
    keep = np.zeros(len(src_np), dtype=bool)
    count = {}
    for i, s in enumerate(src_np):
        c = count.get(s, 0)
        if c < k:
            keep[i] = True
            count[s] = c + 1
    kept_idx = np.flatnonzero(keep)
    edge_index = torch.stack([
        torch.as_tensor(src_np[kept_idx], dtype=torch.long, device=src.device),
        torch.as_tensor(dst_np[kept_idx], dtype=torch.long, device=src.device),
    ])
    return edge_index, w_np[kept_idx]


def row_normalize_with_selfloop(edge_index: torch.Tensor, weight: torch.Tensor, n: int):
    """Add a self-loop (weight = each node's median retained outgoing weight)
    and row-normalize: P = D^-1 A."""
    src, dst = edge_index[0], edge_index[1]
    self_w = torch.full((n,), 1.0, dtype=weight.dtype, device=weight.device)
    for i in range(n):
        m = src == i
        if m.any():
            self_w[i] = weight[m].median()
    self_idx = torch.arange(n, dtype=torch.long, device=edge_index.device)
    full_src = torch.cat([src, self_idx])
    full_dst = torch.cat([dst, self_idx])
    full_w = torch.cat([weight, self_w])

    deg = torch.zeros(n, dtype=weight.dtype, device=weight.device)
    deg.index_add_(0, full_src, full_w)
    deg = deg.clamp_min(1e-8)
    p_w = full_w / deg[full_src]
    return torch.stack([full_src, full_dst]), p_w

Writing /kaggle/working/src/graph_density.py


In [7]:
%%writefile /kaggle/working/src/local_push.py
"""Checkpointed, priority-queue local-push diffusion, with trace-and-replay
differentiation.

Why this is a new module, not an extension of `src/ppr_push.py`
------------------------------------------------------------------
`ppr_push.py`'s `diffuse` pushes the *signal itself* (`r` starts at the
encoder's `z`, and `p[i]` ends up being node i's own diffused representation
directly) -- an efficient trick for transductive node classification that
avoids ever running personalised PageRank per source. Architecture 4R's own
spec does not use that trick: its stage representation is

    h_bar_q^(s) = sum_u p_q^{(s)}(u) * V h_u^{(0)}  /  sum_u p_q^{(s)}(u)

which weights *every node's own initial representation* `h_u^{(0)}` by how
much of query q's personalised-PageRank mass has reached it -- i.e. `p_q` is
a genuine per-query distribution over graph nodes (`e_q`, one unit of mass at
q's own attach node), not a diffused feature vector. That is a real,
per-query local-push computation, one query at a time; batching several
queries into one shared priority queue (an earlier draft of this module did
that) silently breaks each query's own tolerance guarantee, since one
query's large residual can drag a *different* query's node above the queue's
priority threshold before that query's own mass there would justify it. So
this module processes one query at a time; a caller wanting several queries
loops over them explicitly (`arch4r.py`), and that cost -- not this
module's correctness -- is the thing flagged as an open risk in the Phase
0/1 implementation plan (running every training node as its own query is
`O(n / (alpha*eps))` per epoch).

The differentiation problem
----------------------------
The push *order* (which node the priority queue pops next) is a discrete,
data-dependent decision. There is no way to backpropagate through "which
node was popped when" without either pretending it's continuous (a lie) or
fixing it. This module does the latter: run the queue once with gradients
off to produce a `Trace` (the exact sequence of node pops), then *replay*
that exact arithmetic a second time with gradients on, through the edge
weights and the source vector but never through the queue ordering itself.
`tests/test_local_push.py`'s numerical-equivalence test is what makes this
trustworthy: replay must reproduce the non-differentiable pass exactly, to
1e-6, not approximately.
"""

import heapq
from dataclasses import dataclass, field

import numpy as np
import torch

from src.instrument import OpCounter

CHECKPOINT_RATIOS = (float("inf"), 1e-2, 3e-3, 1e-3, 3e-4)


@dataclass
class Checkpoint:
    eps: float


def make_checkpoints(c: float = 1.0) -> list:
    """The 5 cumulative checkpoints, ratios fixed, one tunable global scale `c`."""
    return [Checkpoint(eps=(r if r == float("inf") else r * c)) for r in CHECKPOINT_RATIOS]


def algorithmic_bound(alpha: float, eps: float) -> float:
    """W_q(eps) < 1/(alpha*eps) directed edges for unit source mass.

    Reported alongside the measured count, never substituted for it -- see
    `test_instrumentation_matches_one_real_batch`.
    """
    if eps <= 0 or eps == float("inf"):
        return float("inf")
    return 1.0 / (alpha * eps)


def one_hot_source(query_node_index: int, n: int) -> np.ndarray:
    """e_q: one unit of mass at a single query's attach node -- length n."""
    e = np.zeros(n, dtype=np.float64)
    e[query_node_index] = 1.0
    return e


# ------------------------------------------------------------------- the graph

@dataclass
class _CSRGraph:
    """Compressed out-edge lists -- topology only (v, edge_k), no weight baked
    in. Deliberately decoupled from `edge_weight`: topology (which edges
    exist) is frozen for many training steps (`graph_density.py`'s "at most
    one rebuild after epoch 10" rule), while `edge_weight` -- the GATv2
    scorer's output over the current `z` -- changes every optimizer step.
    Baking weight values into this structure would force a full O(E) python
    rebuild on every call; keeping it weight-free lets one `build_graph(...)`
    call be reused for the whole topology-frozen period, with fresh
    `edge_weight` arrays passed into each push/replay call at zero rebuild
    cost. (Measured: rebuilding this per query, one query per training node
    per epoch, was the dominant cost in a from-scratch transductive-training
    benchmark -- see the implementation plan/handover for the numbers.)"""
    out_edges: list           # out_edges[u] = list of (v, edge_k)
    n: int
    out_deg: np.ndarray = field(default=None)   # out_deg[u] = max(len(out_edges[u]), 1)

    @staticmethod
    def build(edge_index: np.ndarray, n: int) -> "_CSRGraph":
        out = [[] for _ in range(n)]
        for k in range(edge_index.shape[1]):
            u, v = int(edge_index[0, k]), int(edge_index[1, k])
            out[u].append((v, k))
        out_deg = np.array([max(len(e), 1) for e in out], dtype=np.float64)
        return _CSRGraph(out_edges=out, n=n, out_deg=out_deg)


def build_graph(edge_index: np.ndarray, n: int) -> "_CSRGraph":
    """Build the reusable, weight-free adjacency once; pass the result to
    `local_push_trace(..., graph=...)` and `replay_with_grad(..., graph=...)`
    for every query scored against the same (frozen) topology, instead of
    letting each call rebuild it. See `_CSRGraph`'s docstring for why this
    split (topology here, weights passed fresh per call) matters."""
    return _CSRGraph.build(edge_index, n)


# ------------------------------------------------------------------ the trace

@dataclass
class Trace:
    """The exact, replayable record of one non-differentiable push run."""
    events: list = field(default_factory=list)                 # [(node, checkpoint_idx), ...]
    checkpoint_boundaries: list = field(default_factory=list)   # len == n_checkpoints
    visited_edges: set = field(default_factory=set)             # {(u, v, edge_k), ...}, whole run
    n: int = 0
    alpha: float = 0.15

    def edges_visited_between(self, s0: int, s1: int) -> set:
        lo = self.checkpoint_boundaries[s0 - 1] if s0 > 0 else 0
        hi = self.checkpoint_boundaries[s1]
        return {node for node, _ in self.events[lo:hi]}


def trace_churn(trace_a: Trace, trace_b: Trace) -> float:
    """Jaccard overlap of visited-edge sets between two traces of the same
    graph, taken at different points in training. `local_push.py` computes
    this per query; a caller averaging (or taking the median of) several
    queries' churn is what the training procedure's "median overlap" check
    actually means -- this function itself just does the one-trace-pair math."""
    a, b = trace_a.visited_edges, trace_b.visited_edges
    if not a and not b:
        return 1.0
    return len(a & b) / max(len(a | b), 1)


# --------------------------------------------------------- the push, plain

class PushState:
    def __init__(self, p: np.ndarray, r: np.ndarray):
        self.p = p
        self.r = r


def local_push_trace(edge_index: np.ndarray, edge_weight: np.ndarray,
                     source_mass: np.ndarray, alpha: float,
                     checkpoints: list, counter: OpCounter | None = None,
                     graph: "_CSRGraph | None" = None):
    """Non-differentiable, priority-queue-driven local push for ONE query.

    `source_mass` is `e_q`, a length-n distribution over graph nodes
    (typically one-hot, from `one_hot_source`). `p[u]` accumulates how much
    of the query's PPR mass has settled at node u; `p.sum()` is always <= 1.
    Queue key is `r(u) / out_degree(u)`, the standard residual-to-degree
    local-push stopping criterion. Returns (final PushState, list of
    per-checkpoint `p` snapshots, Trace).

    `graph`, if given, is a `build_graph(edge_index, n)` result reused across
    many queries against the same frozen topology -- avoids rebuilding the
    O(E) adjacency on every call (the caller is responsible for passing a
    graph whose topology actually matches `edge_index`; not checked here).
    """
    n = source_mass.shape[0]
    counter = counter or OpCounter()
    graph = graph if graph is not None else _CSRGraph.build(edge_index, n)
    out_deg = graph.out_deg

    p = np.zeros(n, dtype=np.float64)
    r = source_mass.astype(np.float64).copy()
    trace = Trace(n=n, alpha=alpha)
    snapshots = []

    def _priority(u):
        return r[u] / out_deg[u]

    heap = [(-_priority(u), u) for u in range(n) if r[u] > 0]
    heapq.heapify(heap)
    in_heap = {u for _, u in heap}

    for ci, ckpt in enumerate(checkpoints):
        eps = ckpt.eps
        if eps == float("inf"):
            snapshots.append(p.copy())
            trace.checkpoint_boundaries.append(len(trace.events))
            continue
        while heap:
            neg_pri, u = heap[0]
            if -neg_pri <= eps:
                break
            heapq.heappop(heap)
            in_heap.discard(u)
            if _priority(u) <= eps:
                continue  # stale queue entry -- r[u] shrank since it was queued

            counter.count_queue_pop()
            counter.count_node_visit()
            trace.events.append((u, ci))

            if graph.out_edges[u]:
                p[u] += alpha * r[u]
                for v, ek in graph.out_edges[u]:
                    w = edge_weight[ek]
                    r[v] += (1.0 - alpha) * w * r[u]
                    counter.count_edge()
                    counter.count_madd()
                    trace.visited_edges.add((u, v, ek))
                    if r[v] > 0 and v not in in_heap:
                        heapq.heappush(heap, (-_priority(v), v))
                        in_heap.add(v)
            else:
                # A genuinely edgeless node (no self-loop either) has nowhere
                # for the (1-alpha) remainder to go; absorb it all into p[u]
                # rather than silently losing mass. In the real pipeline this
                # never triggers -- `graph_density.row_normalize_with_selfloop`
                # guarantees every node has at least a self-loop -- but this
                # module must not leak mass for an arbitrary input graph.
                p[u] += r[u]
            r[u] = 0.0  # fully settled -- priority(u) is now 0, nothing to requeue

        snapshots.append(p.copy())
        trace.checkpoint_boundaries.append(len(trace.events))

    # `p` alone is the final estimate (standard forward-push: sum(p) + sum(r)
    # == 1 exactly at every point). No further "+alpha*r" adjustment here --
    # that belongs to `ppr_push.py`'s different push-the-signal formulation
    # (every node is simultaneously its own source there), not to this
    # single-query personalised-PageRank push.
    return PushState(p=p, r=r), snapshots, trace


# ------------------------------------------------------ replay, differentiable

class _ReplayTraceFn(torch.autograd.Function):
    """Forward replays `trace` in plain numpy (as cheap as `local_push_trace`
    itself); backward is one hand-derived reverse walk over the same trace,
    not autograd tracking each of the ~thousands of per-event pop/push
    micro-ops. That per-event autograd bookkeeping (individual `index_add`
    calls, one torch node each) was measured to cost ~two orders of
    magnitude more than the equivalent plain-numpy push at real cohort
    scale -- see the implementation plan/handover for the benchmark that
    caught this. Collapsing it to one `Function.apply` call is what makes
    every-training-node-as-its-own-query per epoch tractable.

    The backward derivation (why it's correct, not just fast): every op the
    forward pass performs is an additive accumulation --
    `p[u] += alpha*r[u]` (or the sink's `p[u] += r[u]`) and
    `r[v] += (1-alpha)*w_ek*r[u]` -- so gradients distribute by the ordinary
    sum rule. Walking events in *reverse* chronological order and keeping one
    running per-node adjoint `next_r_bar[u]` (meaning: "the gradient that
    should apply to whatever fed r[u] before its next -- in forward time --
    pop") is sufficient: at a pop of `u`, `next_r_bar[u]` (already resolved,
    since anything at a later forward time was visited earlier in this
    reverse walk) plus the checkpoint-snapshot gradients accumulated so far
    (`p_bar`) give this pop's own `grad_r_before`, which both (a) attributes
    to every incoming edge that fed it -- accumulating `edge_weight`'s
    gradient there -- and (b) overwrites `next_r_bar[u]` for the next
    (earlier) increment into `u`. Whatever remains in `next_r_bar` once every
    event has been walked is exactly the gradient into `source_mass`, since
    the initial deposit is just the earliest increment in the same additive
    chain. `p_bar` itself is built by adding each checkpoint's own output
    gradient before entering the segment of events that checkpoint's
    snapshot actually includes (working from the last checkpoint back to the
    first), matching that a snapshot is the sum of every increment at or
    before its own boundary.
    """

    @staticmethod
    def forward(ctx, edge_weight: torch.Tensor, source_mass: torch.Tensor,
               trace: Trace, graph: "_CSRGraph", alpha: float, n_checkpoints: int):
        w_np = edge_weight.detach().cpu().numpy().astype(np.float64)
        s_np = source_mass.detach().cpu().numpy().astype(np.float64)
        n = s_np.shape[0]

        p = np.zeros(n, dtype=np.float64)
        r = s_np.copy()
        n_events = len(trace.events)
        r_before = np.empty(n_events, dtype=np.float64)
        is_sink = np.empty(n_events, dtype=bool)

        snapshots = []
        boundary_iter = iter(trace.checkpoint_boundaries)
        next_boundary = next(boundary_iter, n_events)
        idx = 0
        for _ in range(n_checkpoints):
            while idx < next_boundary:
                u, _ci = trace.events[idx]
                rb = r[u]
                r_before[idx] = rb
                edges_u = graph.out_edges[u]
                if edges_u:
                    is_sink[idx] = False
                    p[u] += alpha * rb
                    for v, ek in edges_u:
                        r[v] += (1.0 - alpha) * w_np[ek] * rb
                else:
                    is_sink[idx] = True
                    p[u] += rb
                r[u] = 0.0
                idx += 1
            snapshots.append(p.copy())
            next_boundary = next(boundary_iter, n_events)

        ctx.trace, ctx.graph, ctx.alpha = trace, graph, alpha
        ctx.r_before, ctx.is_sink, ctx.w_np = r_before, is_sink, w_np
        ctx.ew_dtype, ctx.ew_device = edge_weight.dtype, edge_weight.device
        ctx.sm_dtype, ctx.sm_device = source_mass.dtype, source_mass.device

        y = np.stack(snapshots, axis=0)  # [n_checkpoints, n]
        return torch.tensor(y, dtype=edge_weight.dtype, device=edge_weight.device)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        trace, graph, alpha = ctx.trace, ctx.graph, ctx.alpha
        r_before, is_sink, w_np = ctx.r_before, ctx.is_sink, ctx.w_np
        events, boundaries = trace.events, trace.checkpoint_boundaries

        go = grad_output.detach().cpu().numpy().astype(np.float64)  # [K, n]
        n = go.shape[1]

        p_bar = np.zeros(n, dtype=np.float64)
        next_r_bar = np.zeros(n, dtype=np.float64)
        grad_w = np.zeros_like(w_np, dtype=np.float64)

        for s in range(go.shape[0] - 1, -1, -1):
            p_bar += go[s]
            seg_start = boundaries[s - 1] if s > 0 else 0
            seg_end = boundaries[s]
            for e_idx in range(seg_end - 1, seg_start - 1, -1):
                u, _ci = events[e_idx]
                rb = r_before[e_idx]
                if not is_sink[e_idx]:
                    grad_rb = alpha * p_bar[u]
                    for v, ek in graph.out_edges[u]:
                        rb_v = next_r_bar[v]
                        if rb_v != 0.0:
                            grad_w[ek] += rb_v * (1.0 - alpha) * rb
                            grad_rb += rb_v * (1.0 - alpha) * w_np[ek]
                else:
                    grad_rb = p_bar[u]
                next_r_bar[u] = grad_rb

        grad_edge_weight = torch.tensor(grad_w, dtype=ctx.ew_dtype, device=ctx.ew_device)
        grad_source_mass = torch.tensor(next_r_bar, dtype=ctx.sm_dtype, device=ctx.sm_device)
        return grad_edge_weight, grad_source_mass, None, None, None, None


def replay_with_grad(trace: Trace, edge_index: np.ndarray, edge_weight: torch.Tensor,
                     source_mass: torch.Tensor, alpha: float, n_checkpoints: int,
                     graph: "_CSRGraph | None" = None):
    """Replay `trace`'s exact node-pop order for ONE query with gradients
    flowing through `edge_weight` and `source_mass`, never through the
    (fixed, detached) pop order.

    Returns (final p [n] tensor with grad, list of per-checkpoint p
    snapshots with grad). Must reproduce `local_push_trace`'s own snapshots
    to 1e-6 when `edge_weight`/`source_mass` hold the same values used to
    build the trace -- that equality is the test that makes this
    trustworthy (`test_replay_matches_nondifferentiable_inference_to_1e6`).

    `graph`, if given, is a `build_graph(edge_index, n)` result reused across
    many queries against the same frozen topology -- see `_CSRGraph`.
    """
    if graph is None:
        graph = _CSRGraph.build(edge_index, trace.n)
    y = _ReplayTraceFn.apply(edge_weight, source_mass, trace, graph, alpha, n_checkpoints)
    snapshots = list(y.unbind(0))
    return snapshots[-1], snapshots


# ------------------------------------------------------------ stage output

def stage_context_from_Vh0(Vh0: torch.Tensor, p_stage: torch.Tensor) -> torch.Tensor:
    """Same as `stage_context`, but takes `V(h0)` already applied.

    `V(h0)` doesn't depend on the query or the checkpoint -- callers scoring
    many (query, checkpoint) pairs against the same `h0`/`V` (e.g. arch4r.py's
    per-query diffusion loop) should compute it once and reuse it here,
    rather than calling `stage_context` and re-running `V` on every pair
    (each redundant call keeps its own [n, d_hidden] activation alive for
    backward(), which is what OOM'd a 16 GB GPU scoring thousands of queries
    x 5 checkpoints).
    """
    weighted = (p_stage.unsqueeze(1) * Vh0).sum(dim=0)
    return weighted / (p_stage.sum() + 1e-8)


def stage_context(h0: torch.Tensor, p_stage: torch.Tensor, V: torch.nn.Module) -> torch.Tensor:
    """h_bar_q^(s) = (sum_u p_q^{(s)}(u) V h_u^{(0)}) / (sum_u p_q^{(s)}(u) + eps).

    `h0` is [n, d_in] (every graph node's own initial representation),
    `p_stage` is [n] (one query's PPR mass at checkpoint s). Returns [d_hidden].
    """
    return stage_context_from_Vh0(V(h0), p_stage)


def stage_representation(h0_q: torch.Tensor, h_bar_q: torch.Tensor,
                         w_self: torch.nn.Module, w_diff: torch.nn.Module,
                         w_int: torch.nn.Module, ln: torch.nn.Module) -> torch.Tensor:
    """g_q^(s) = LN(W_self h0_q + W_diff h_bar_q + W_int (h0_q * h_bar_q) + b).

    `h0_q` is query q's own initial representation [d_in]; `h_bar_q` is
    `stage_context`'s output for the same query and checkpoint. The
    elementwise product `h0_q * h_bar_q` requires `V`'s output dimension to
    equal `d_in` -- the caller (`arch4r.py`) is responsible for that, e.g.
    by giving `V` the same width as the fusion layer's output.
    """
    combined = w_self(h0_q) + w_diff(h_bar_q) + w_int(h0_q * h_bar_q)
    return ln(combined)

Writing /kaggle/working/src/local_push.py


In [8]:
%%writefile /kaggle/working/src/arch4r.py
"""Architecture 4R model wiring: encoder + density-adaptive graph + local-push
diffusion, in three ablation arms sharing every learned component except the
topology's own `k`.

Why the three arms are one model class, not three
----------------------------------------------------
`encoder_only`, `fixed_k`, and `adaptive_k` must share the encoder, the
GATv2-style edge scorer, and the view-reliability fusion -- the plan's own
required invariant (`test_fixed_k_and_adaptive_k_share_every_component_
except_k`). This module gets that for free by construction: `Arch4RModel`
always builds every graph-dependent submodule regardless of `mode`, and
`mode=="encoder_only"` simply never calls them (no `attached_graph` is
consulted at all in that branch -- not "consulted but ignored", genuinely
untouched, which is what `test_encoder_only_arm_has_no_graph_dependency`
checks via a no-gradient assertion). `fixed_k` vs `adaptive_k` never appear
as a code branch inside this module at all: which `k` a given arm uses is a
property of the `graph_density.ViewGraphConfig` that built the
`Arch4RGraphBatch` fed into `forward`, not of anything this model owns.

Cold start / G_q=0 (no graph) fallback
----------------------------------------
A query with `graph_available[i] == False` (or, in `encoder_only` mode,
every query) gets `h_bar_q^(s) = h0_q` at every checkpoint -- there is no
diffused context to weight by, so the stage representation degenerates to
using the query's own encoded representation at every stage. This makes
`encoder_only`'s output structurally identical to a graph-available query
whose graph happened to return nothing, which is the intended behaviour
(the plan calls `encoder_only` "also...the real code path other arms fall
back into").

Transductive training's per-query cost (see `src/local_push.py`)
--------------------------------------------------------------------
Training is transductive over the training reference graph: every train
node is simultaneously a member of the graph and, one at a time, a query
scored against it. A from-scratch benchmark (see the implementation
handover) found this looked infeasible at real cohort scale purely from two
redundant per-query rebuild costs in `local_push.py`, now fixed there
(`build_graph(...)` built once per `forward()` call and reused for every
query in `attached_graph.query_idx`, not rebuilt per query) -- this module
relies on that fix by building the fused adjacency and its `build_graph(...)`
topology exactly once per `forward()` call.
"""

from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.encoder_mv import MultiViewEncoder, VIEW_OUT_DIMS
from src.graph_density import (EdgeScorerGATv2, ViewReliability, fuse_views,
                               row_normalize_with_selfloop)
from src.instrument import OpCounter
from src.local_push import (build_graph, local_push_trace, one_hot_source,
                            replay_with_grad, stage_context_from_Vh0,
                            stage_representation)

MODES = ("encoder_only", "fixed_k", "adaptive_k")


@dataclass
class Arch4RGraphBatch:
    """Everything `Arch4RModel.forward` needs to run local-push diffusion for
    a batch of query nodes against a fixed graph-node set.

    `per_view_edges[name]` is `(src, dst, base_weight)`, indices into the
    COMBINED node space `[0, n_nodes)` that `view_x`/`view_observed` (passed
    to `forward` alongside this batch) were encoded over.

    Two constructions, via the helpers below:
      - `transductive_batch(...)`: `n_nodes == n_train`, every train node is
        both a graph member and its own query (`query_idx ==
        arange(n_train)`), edges are the frozen train-train topology.
      - `attach_batch(...)`: `n_nodes == n_train + n_query`, held-out queries
        are appended after the frozen train block, edges are the train-train
        topology PLUS `ReferenceGraph.attach(...)`'s query->train edges (a
        query must be able to diffuse through the whole reference graph, not
        just reach its first ring).
    """
    per_view_edges: dict
    n_nodes: int
    query_idx: np.ndarray
    graph_available: np.ndarray


def transductive_batch(ref_graph, k_min: int | None = None) -> Arch4RGraphBatch:
    """Build the transductive-training batch: every train node is
    simultaneously a graph node and its own query, over the frozen
    train-train edges `graph_density.build_reference_graph` already built."""
    per_view_edges = {
        name: (v.edge_index_ref[0], v.edge_index_ref[1], v.edge_weight_ref)
        for name, v in ref_graph.views.items()
    }
    n_train = next(iter(ref_graph.views.values())).X_train.shape[0]
    available = ref_graph.train_graph_available(k_min=k_min)
    return Arch4RGraphBatch(per_view_edges=per_view_edges, n_nodes=n_train,
                            query_idx=np.arange(n_train), graph_available=available)


def attach_batch(ref_graph, attached, n_train: int) -> Arch4RGraphBatch:
    """Build the held-out-evaluation batch: `attached` (a
    `graph_density.AttachedGraph` from `ReferenceGraph.attach(...)`) is
    appended after the frozen train block so held-out queries can diffuse
    through the whole reference graph."""
    n_query = attached.n_query
    n_nodes = n_train + n_query
    per_view_edges = {}
    for name, v in ref_graph.views.items():
        tt_src, tt_dst, tt_w = v.edge_index_ref[0], v.edge_index_ref[1], v.edge_weight_ref
        if name in attached.per_view_edges:
            q_src, q_dst, q_w = attached.per_view_edges[name]
            q_src = q_src + n_train  # shift query-local indices into combined space
        else:
            q_src = np.zeros(0, dtype=int)
            q_dst = np.zeros(0, dtype=int)
            q_w = np.zeros(0, dtype=tt_w.dtype)
        per_view_edges[name] = (
            np.concatenate([tt_src, q_src]),
            np.concatenate([tt_dst, q_dst]),
            np.concatenate([tt_w, q_w]),
        )
    query_idx = np.arange(n_train, n_nodes)
    return Arch4RGraphBatch(per_view_edges=per_view_edges, n_nodes=n_nodes,
                            query_idx=query_idx, graph_available=attached.graph_available)


class Arch4RModel(nn.Module):
    def __init__(self, view_in_dims: dict[str, int], cfg: dict, mode: str):
        super().__init__()
        if mode not in MODES:
            raise ValueError(f"mode must be one of {MODES}, got {mode!r}")
        self.mode = mode
        self.cfg = dict(cfg)

        fusion_dim = self.cfg.get("fusion_dim", 64)
        stage_hidden = self.cfg.get("stage_hidden", 32)
        out_dims = self.cfg.get("view_out_dims", VIEW_OUT_DIMS)
        self.alpha = self.cfg.get("alpha", 0.15)
        self.top_k_keep = self.cfg.get("top_k_keep", 64)

        self.encoder = MultiViewEncoder(view_in_dims, fusion_dim=fusion_dim,
                                        dropout=self.cfg.get("dropout", 0.15),
                                        out_dims=out_dims)
        self.view_names = list(self.encoder.view_names)

        # Always built, regardless of mode -- see module docstring.
        z_dims = {v: out_dims[v] for v in self.view_names}
        self.scorers = nn.ModuleDict({
            v: EdgeScorerGATv2(z_dims[v], hidden=16) for v in self.view_names
        })
        self.reliability = ViewReliability(z_dims)
        self.V = nn.Linear(fusion_dim, fusion_dim)
        self.w_self = nn.Linear(fusion_dim, stage_hidden)
        self.w_diff = nn.Linear(fusion_dim, stage_hidden)
        self.w_int = nn.Linear(fusion_dim, stage_hidden)
        self.stage_ln = nn.LayerNorm(stage_hidden)
        self.stage_head = nn.Linear(stage_hidden, 1)
        # Graph-independent by construction (operates on h0 alone) so it
        # stays available even in encoder_only / cold-start fallback.
        self.error_head = nn.Linear(fusion_dim, 1)

    def n_trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ------------------------------------------------------------- forward

    def forward(self, view_x: dict[str, torch.Tensor], view_observed: dict[str, torch.Tensor],
               attached_graph: Arch4RGraphBatch | None, checkpoints: list) -> dict:
        enc_out = self.encoder(view_x, view_observed)
        h0 = enc_out["h0"]
        n_nodes = h0.shape[0]
        n_checkpoints = len(checkpoints)
        counter = OpCounter()

        if attached_graph is not None:
            query_idx = np.asarray(attached_graph.query_idx)
        else:
            query_idx = np.arange(n_nodes)
        n_query = len(query_idx)

        use_graph = (self.mode != "encoder_only") and (attached_graph is not None)

        query_idx_t = torch.as_tensor(query_idx, dtype=torch.long, device=h0.device)
        h0_q = h0[query_idx_t]
        error_head_raw = self.error_head(h0_q).squeeze(-1)

        if not use_graph:
            g = stage_representation(h0_q, h0_q, self.w_self, self.w_diff, self.w_int,
                                     self.stage_ln)
            logit = self.stage_head(g).squeeze(-1)
            stage_logits = [logit for _ in range(n_checkpoints)]
            return {"h0": h0_q, "stage_logits": stage_logits,
                   "error_head_raw": error_head_raw, "ledger": counter.snapshot()}

        graph_available = np.asarray(attached_graph.graph_available)
        view_z = enc_out["view_z"]
        reliability_pi = self.reliability(view_z, enc_out["reliability_raw"])

        scorer_output = {}
        for v, (src, dst, _base_w) in attached_graph.per_view_edges.items():
            if len(src) == 0:
                scorer_output[v] = torch.zeros(0, device=h0.device)
                continue
            src_t = torch.as_tensor(src, dtype=torch.long, device=h0.device)
            dst_t = torch.as_tensor(dst, dtype=torch.long, device=h0.device)
            scorer_output[v] = self.scorers[v](view_z[v][src_t], view_z[v][dst_t])

        edge_index, edge_weight = fuse_views(attached_graph.per_view_edges, reliability_pi,
                                             scorer_output, n_nodes=n_nodes,
                                             top_k_keep=self.top_k_keep)
        full_edge_index, full_p_weight = row_normalize_with_selfloop(edge_index, edge_weight, n_nodes)

        edge_index_np = full_edge_index.detach().cpu().numpy()
        edge_weight_np = full_p_weight.detach().cpu().numpy()
        # Built once, reused for every query below -- the fix that makes
        # scoring every training node as its own query per epoch tractable.
        shared_graph = build_graph(edge_index_np, n_nodes)
        # V(h0) doesn't depend on the query or the checkpoint -- computing it
        # once here instead of inside the loop below avoids keeping one
        # [n_nodes, d_hidden] activation alive per (query, checkpoint) pair,
        # which is what OOM'd a 16 GB GPU even after batching the query loop.
        Vh0 = self.V(h0)

        stage_logits_rows = [[] for _ in range(n_checkpoints)]
        for local_i, q in enumerate(query_idx):
            if not graph_available[local_i]:
                h_bar = h0[q]
                g = stage_representation(h0[q], h_bar, self.w_self, self.w_diff,
                                         self.w_int, self.stage_ln)
                logit = self.stage_head(g).squeeze(-1)
                for s in range(n_checkpoints):
                    stage_logits_rows[s].append(logit)
                continue

            e_q = one_hot_source(int(q), n_nodes)
            _, _, trace = local_push_trace(edge_index_np, edge_weight_np, e_q,
                                           alpha=self.alpha, checkpoints=checkpoints,
                                           counter=counter, graph=shared_graph)
            source_t = torch.tensor(e_q, dtype=full_p_weight.dtype, device=h0.device)
            _, snapshots = replay_with_grad(trace, edge_index_np, full_p_weight, source_t,
                                            alpha=self.alpha, n_checkpoints=n_checkpoints,
                                            graph=shared_graph)
            for s in range(n_checkpoints):
                h_bar = stage_context_from_Vh0(Vh0, snapshots[s])
                g = stage_representation(h0[q], h_bar, self.w_self, self.w_diff,
                                         self.w_int, self.stage_ln)
                logit = self.stage_head(g).squeeze(-1)
                stage_logits_rows[s].append(logit)

        stage_logits = [torch.stack(rows) for rows in stage_logits_rows]
        return {"h0": h0_q, "stage_logits": stage_logits,
               "error_head_raw": error_head_raw, "ledger": counter.snapshot()}


# ------------------------------------------------------------------- loss

def arch4r_loss(out: dict, y: torch.Tensor, mask: torch.Tensor, cfg: dict,
                lam_surv: float = 0.0):
    """Binary cross-entropy over every checkpoint's stage logits, averaged
    over checkpoints (deep supervision across exits) then over masked rows.

    `lam_surv` is reserved for the Phase 2/3 survival head -- Phase 1 is
    binary-only, so it must be left at 0.0 here; the parameter exists only
    so this signature won't need to break when that head is added.
    """
    if lam_surv != 0.0:
        raise NotImplementedError("survival head is Phase 2/3 scope; lam_surv must be 0.0")
    y_m = y[mask].float()
    total = 0.0
    info = {}
    for s, logits in enumerate(out["stage_logits"]):
        l = F.binary_cross_entropy_with_logits(logits[mask], y_m)
        info[f"stage{s}_bce"] = float(l.detach())
        total = total + l
    total = total / len(out["stage_logits"])
    info["total"] = float(total.detach())
    return total, info

Writing /kaggle/working/src/arch4r.py


## Splits -- verbatim copies of `data_prep.make_splits` and `cascade.split_validation_3way`

In [ ]:
def make_splits(y, seed, fracs=(0.6, 0.2, 0.2)):
    rng = np.random.default_rng(seed)
    n = len(y)
    train, val, test = (np.zeros(n, dtype=bool) for _ in range(3))
    for cls in np.unique(y):
        idx = np.flatnonzero(y == cls)
        rng.shuffle(idx)
        n_tr = int(round(fracs[0] * len(idx))); n_va = int(round(fracs[1] * len(idx)))
        train[idx[:n_tr]] = True; val[idx[n_tr:n_tr + n_va]] = True; test[idx[n_tr + n_va:]] = True
    return train, val, test


def split_validation_3way(val_mask, y, seed, fracs=(0.4, 0.3, 0.3)):
    rng = np.random.default_rng(seed + 992)
    parts = [np.zeros_like(val_mask) for _ in range(3)]
    idx = np.flatnonzero(val_mask)
    for cls in (0, 1):
        c = rng.permutation(idx[y[idx] == cls])
        cut1 = int(round(fracs[0] * len(c))); cut2 = cut1 + int(round(fracs[1] * len(c)))
        parts[0][c[:cut1]] = True; parts[1][c[cut1:cut2]] = True; parts[2][c[cut2:]] = True
    return parts


def auroc(y, p):
    return float(roc_auc_score(y, p))


def auprc(y, p):
    return float(average_precision_score(y, p))


def set_seed(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## Preprocessing variants

`iqr` is exactly Arch4R's `encoder_mv.fit_view_preprocessors` (median impute,
clip to Q1-1.5·IQR / Q3+1.5·IQR, scale by IQR) -- which clips away the lab/
vital extremes (e.g. `lab_lactate_max`) that carry most of the mortality
signal. `pct` clips at the 0.5/99.5 train percentiles instead; `quantile`
maps each continuous column through a train-fit quantile transform to a
standard normal. Binary columns are left as 0/1 in every variant. With
`indicators=True`, one 0/1 missingness column is appended per raw column that
is ever missing in train, plus each view's observed fraction.

In [ ]:
from sklearn.preprocessing import QuantileTransformer


def prep_views(X, views, tr, kind, indicators):
    blocks, slices, start = [], {}, 0
    for v, idx in views.items():
        x = X[:, idx].astype(np.float64)
        miss = np.isnan(x)
        allmiss = miss[tr].all(0)
        x, miss = x[:, ~allmiss], miss[:, ~allmiss]
        obs_tr = np.where(miss[tr], np.nan, x[tr])
        binary = np.array([np.isin(c[~np.isnan(c)], (0.0, 1.0)).all() for c in obs_tr.T])
        med = np.nan_to_num(np.nanmedian(obs_tr, 0))
        z = np.zeros_like(x)
        cont = ~binary
        if kind == "quantile":
            qt = QuantileTransformer(n_quantiles=256, output_distribution="normal",
                                     subsample=10**9, random_state=0)
            qt.fit(obs_tr[:, cont])
            zc = qt.transform(x[:, cont])
            z[:, cont] = np.nan_to_num(zc, nan=0.0)
        else:
            q1, q3 = np.nanpercentile(obs_tr, 25, 0), np.nanpercentile(obs_tr, 75, 0)
            iqr = np.where(np.isnan(q3 - q1) | (q3 - q1 < 1e-6), 1.0, q3 - q1)
            if kind == "iqr":
                lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            else:  # pct
                lo, hi = np.nanpercentile(obs_tr, 0.5, 0), np.nanpercentile(obs_tr, 99.5, 0)
            xc = np.clip(np.where(miss, med, x), lo, hi)
            z[:, cont] = ((xc - med) / iqr)[:, cont]
        xb = np.where(miss, med, x)[:, binary]
        z[:, binary] = xb - xb[tr].mean(0)
        parts = [z]
        if indicators:
            keep = miss[tr].any(0)
            parts += [miss[:, keep].astype(np.float64), 1.0 - miss.mean(1, keepdims=True)]
        zz = np.concatenate(parts, 1).astype(np.float32)
        slices[v] = (start, start + zz.shape[1]); start += zz.shape[1]
        blocks.append(zz)
    return np.concatenate(blocks, 1), slices

## Shared model pieces + training loop (AdamW, early stopping on model_val AUROC)

In [ ]:
class ViewEnc(nn.Module):
    def __init__(s, i, o, hid, p):
        super().__init__()
        s.fc1, s.fc2, s.drop, s.ln = nn.Linear(i, hid), nn.Linear(hid, o), nn.Dropout(p), nn.LayerNorm(o)

    def forward(s, x):
        return s.ln(s.fc2(s.drop(F.gelu(s.fc1(x)))))


class Encoder(nn.Module):
    """Multi-view encoder: one MLP per view, residual fusion -> h [n, d]."""

    def __init__(s, slices, d=128, hid=128, vout=32, p=0.2):
        super().__init__()
        s.slices = slices
        s.enc = nn.ModuleDict({v: ViewEnc(b - a, vout, hid, p) for v, (a, b) in slices.items()})
        cat = vout * len(slices)
        s.skip = nn.Linear(cat, d)
        s.fuse = nn.Sequential(nn.Linear(cat, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d)

    def forward(s, x):
        z = torch.cat([s.enc[v](x[:, a:b]) for v, (a, b) in s.slices.items()], 1)
        return s.ln(s.skip(z) + s.fuse(z))


def head(i, p):
    return nn.Sequential(nn.Linear(i, i), nn.GELU(), nn.Dropout(p), nn.Linear(i, 1))


class EncoderOnly(nn.Module):
    def __init__(s, slices, d=128, p=0.2):
        super().__init__()
        s.enc, s.h0 = Encoder(slices, d=d, p=p), head(d, p)

    def forward(s, x):
        return [s.h0(s.enc(x)).squeeze(-1)]


def fit_minibatch(model, forward_fn, tr_idx, y_t, val_fn, lr=1e-3, wd=1e-4, bs=512,
                  max_ep=300, patience=20, seed=0, restore_best=True):
    """Generic mini-batch loop. `forward_fn(idx, train=True)` -> list of exit
    logits for rows idx; loss = mean BCE over exits (deep supervision, as in
    arch4r_loss). `val_fn()` -> list of per-exit model_val AUROCs; the best
    epoch by the LAST exit's val AUROC is restored."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    rng = np.random.default_rng(seed)
    best, best_state, bad, best_ep, t0 = -1.0, None, 0, 0, time.time()
    for ep in range(max_ep):
        model.train()
        perm = rng.permutation(tr_idx)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            outs = forward_fn(b, train=True)
            loss = sum(F.binary_cross_entropy_with_logits(o, y_t[b]) for o in outs) / len(outs)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        v = val_fn()[-1]
        if v > best:
            best, bad, best_ep = v, 0, ep
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if restore_best:
        model.load_state_dict(best_state)
    return {"val_auroc": best, "best_epoch": best_ep, "epochs_run": ep + 1,
            "restored_best": restore_best, "train_s": time.time() - t0}


@torch.no_grad()
def predict_rows(forward_fn, model, idx, bs=2048):
    model.eval()
    outs = None
    for s in range(0, len(idx), bs):
        o = [t.float().cpu().numpy() for t in forward_fn(idx[s:s + bs], train=False)]
        outs = o if outs is None else [np.concatenate([a, b]) for a, b in zip(outs, o)]
    return outs

## G1 — TabR-style label-aware retrieval over a learned metric

For a query batch, keys of ALL training patients are computed without
gradient, the m nearest (L2 in key space) are selected -- **the query itself
is excluded** when it is a training patient -- and only the selected
neighbours are re-encoded with gradient. Each neighbour contributes
`LabelEmb(y_j) + T(k_q - k_j)` weighted by softmax(-distance); held-out
queries retrieve from the training pool only, so no validation/test label is
ever an input.

In [ ]:
class Retrieval(nn.Module):
    def __init__(s, slices, d=128, m=96, p=0.2, ctx_drop=0.2):
        super().__init__()
        s.enc, s.m = Encoder(slices, d=d, p=p), m
        s.K = nn.Linear(d, d)
        s.lab = nn.Embedding(2, d)
        s.T = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, d, bias=False))
        s.block = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d), nn.ReLU(), nn.Dropout(p),
                                nn.Linear(2 * d, d))
        s.ctx_drop = nn.Dropout(ctx_drop)
        s.h0, s.h1 = head(d, p), head(d, p)

    def forward(s, xq, q_pool_pos, x_pool, y_pool):
        hq = s.enc(xq)
        kq = s.K(hq)
        with torch.no_grad():
            kp = torch.cat([s.K(s.enc(x_pool[i:i + 4096])) for i in range(0, len(x_pool), 4096)])
            dist = torch.cdist(kq.detach(), kp) ** 2
            if q_pool_pos is not None:  # training queries: never retrieve yourself
                dist[torch.arange(len(xq), device=xq.device), q_pool_pos] = float("inf")
            nbr = dist.topk(s.m, dim=1, largest=False).indices          # [B, m]
        kn = s.K(s.enc(x_pool[nbr.reshape(-1)])).reshape(len(xq), s.m, -1)
        w = s.ctx_drop(torch.softmax(-((kq[:, None, :] - kn) ** 2).sum(-1), dim=1))
        v = s.lab(y_pool[nbr]) + s.T(kq[:, None, :] - kn)
        h1 = hq + (w[..., None] * v).sum(1)
        h1 = h1 + s.block(h1)
        return [s.h0(hq).squeeze(-1), s.h1(h1).squeeze(-1)]

## G2 — masked-label PPR diffusion on a learned kNN graph (Arch4R lineage)

Every node's out-edges go to its `kk` nearest *training* nodes (self
excluded) in a learned cosine key space, weighted by softmax(sim/τ) -- so
held-out nodes read from the training graph but no training node ever reads
a held-out node (inductive, no held-out-to-held-out edges, exactly Arch4R's
attach discipline). Each node carries `[V h, ỹ, vis]`, where `vis`=1 marks a
visible training label and `ỹ = y·vis`. APPNP diffusion
`Z ← (1-α)·A·Z + α·Z0`; at checkpoints t ∈ `steps`, exit s reads
`[h, Z_emb, Z_ỹ/Z_vis, Z_vis]`. Training uses the label trick: each step a
random `keep` fraction of training labels is visible and the loss is taken
ONLY on the hidden ones -- a patient's own label never enters its own score.
At evaluation every training label is visible; held-out labels never are.

In [ ]:
class LabelPPR(nn.Module):
    def __init__(s, slices, d=128, kk=16, alpha=0.15, steps=(1, 2, 4, 8, 16), p=0.2, kdim=32):
        super().__init__()
        s.enc, s.kk, s.alpha, s.steps = Encoder(slices, d=d, p=p), kk, alpha, tuple(steps)
        s.K, s.V = nn.Linear(d, kdim), nn.Linear(d, d)
        s.log_tau = nn.Parameter(torch.tensor(math.log(0.1)))
        s.h0 = head(d, p)
        s.hs = nn.ModuleList([head(2 * d + 2, p) for _ in steps])

    def forward(s, x, n_pool, y_vis, vis):
        """x: [N, D] with rows [0, n_pool) = training pool. y_vis/vis: [N]
        (zero outside visible training rows). Returns exit logits for all N."""
        h = s.enc(x)
        k = F.normalize(s.K(h), dim=-1)
        N = len(x)
        with torch.no_grad():
            nbr = torch.empty(N, s.kk, dtype=torch.long, device=x.device)
            kp = k[:n_pool].detach()
            for i in range(0, N, 4096):
                sim = k[i:i + 4096].detach() @ kp.T
                r = torch.arange(i, min(i + 4096, N), device=x.device)
                own = r < n_pool
                sim[own.nonzero().squeeze(1), r[own]] = -float("inf")
                nbr[i:i + 4096] = sim.topk(s.kk, dim=1).indices
        A = torch.softmax((k[:, None, :] * k[nbr]).sum(-1) / s.log_tau.exp(), dim=1)  # [N, kk]
        Z0 = torch.cat([s.V(h), y_vis[:, None], vis[:, None]], 1)
        Z, outs, si = Z0, [s.h0(h).squeeze(-1)], 0
        for t in range(1, max(s.steps) + 1):
            Z = (1 - s.alpha) * (A[..., None] * Z[nbr]).sum(1) + s.alpha * Z0
            if t in s.steps:
                rate = Z[:, -2:-1] / (Z[:, -1:] + 1e-6)
                outs.append(s.hs[si](torch.cat([h, Z[:, :-2], rate, Z[:, -1:]], 1)).squeeze(-1))
                si += 1
        return outs


def fit_labelppr(model, Xt, y_t, tr_idx, mv_idx, keep=0.5, lr=1e-3, wd=1e-4,
                 max_steps=3000, eval_every=20, patience=25, seed=0):
    """Full-batch over the training graph; label-trick masking resampled every step."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    g = torch.Generator(device=Xt.device).manual_seed(seed)
    x_tr, y_tr = Xt[tr_idx], y_t[tr_idx]
    n = len(tr_idx)
    best, best_state, bad, best_step, t0 = -1.0, None, 0, 0, time.time()
    for step in range(1, max_steps + 1):
        model.train()
        vis = (torch.rand(n, generator=g, device=Xt.device) < keep).float()
        outs = model(x_tr, n, y_tr * vis, vis)
        hid = vis == 0
        loss = sum(F.binary_cross_entropy_with_logits(o[hid], y_tr[hid]) for o in outs) / len(outs)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        if step % eval_every == 0:
            v = labelppr_predict(model, Xt, y_t, tr_idx, mv_idx)
            a = auroc(Y_NP[mv_idx], v[-1])
            if a > best:
                best, bad, best_step = a, 0, step
                best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    model.load_state_dict(best_state)
    return {"val_auroc": best, "best_step": best_step, "steps_run": step, "train_s": time.time() - t0}


@torch.no_grad()
def labelppr_predict(model, Xt, y_t, tr_idx, q_idx):
    model.eval()
    n = len(tr_idx)
    x = torch.cat([Xt[tr_idx], Xt[q_idx]])
    vis = torch.cat([torch.ones(n, device=Xt.device), torch.zeros(len(q_idx), device=Xt.device)])
    y_vis = torch.cat([y_t[tr_idx], torch.zeros(len(q_idx), device=Xt.device)])
    outs = model(x, n, y_vis, vis)
    return [o[n:].float().cpu().numpy() for o in outs]

## Smoke test on synthetic data (fails fast, before the ~7-min cohort build)

In [ ]:
_sl = {"a": (0, 7), "b": (7, 12)}
_x = torch.randn(300, 12, device=DEV)
_y = (torch.rand(300, device=DEV) < 0.2).float()
_g1 = Retrieval(_sl, m=8).to(DEV)
_o = _g1(_x[:32], torch.arange(32, device=DEV), _x[:200], _y[:200].long())
sum(o.mean() for o in _o).backward()
_g2 = LabelPPR(_sl, kk=4).to(DEV)
_vis = (torch.rand(300, device=DEV) < 0.5).float()
_o = _g2(_x, 200, _y * _vis, _vis)
sum(o.mean() for o in _o).backward()
assert len(_o) == 6 and _o[-1].shape == (300,)
_Xs, _ss = prep_views(np.where(np.random.rand(300, 12) < 0.3, np.nan, np.random.randn(300, 12)),
                      {"a": np.arange(7), "b": np.arange(7, 12)}, np.arange(300) < 200, "quantile", True)
assert np.isfinite(_Xs).all()
print("[v2] smoke test OK", flush=True)

## Cohort (same definition/signature as notebooks 09-11: expect `cohort_97de93de81`)

In [ ]:
def _find_mimic_root(start, max_depth=5):
    for dirpath, dirnames, _files in os.walk(start):
        if "hosp" in dirnames:
            base = os.path.basename(dirpath)
            return os.path.dirname(dirpath) if base in ("mimic-iv-3.1", "mimic-iv") else dirpath
        if dirpath[len(start):].count(os.sep) >= max_depth:
            dirnames[:] = []
    return None

RAW = _find_mimic_root("/kaggle/input") if os.path.isdir("/kaggle/input") else None
if RAW is None:
    import kagglehub
    _dl = kagglehub.dataset_download("akshaybe/updated-mimic-iv")
    RAW = _find_mimic_root(_dl, max_depth=3) or _dl
print("[v2] MIMIC root:", RAW, flush=True)

try:
    import duckdb  # noqa: F401  (mimic_iv falls back to a much slower pandas scan without it)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

from src import mimic_iv
CFG = {"seed": 42, "data": {
    "source": "mimic", "task": "mortality", "raw_dir": RAW,
    "cache_dir": f"{WORK}/processed", "obs_window_hours": 24, "min_los_hours": 28,
    "max_patients": None, "cohort_seed": 42, "force_rebuild": False,
    "blocks": ["demographics", "comorbidity", "labs", "vitals", "treatment"],
    "require_nonempty_blocks": ["vitals"]}}
print("[v2] cohort signature:", mimic_iv.cohort_signature(CFG), flush=True)
t0 = time.time()
X_RAW, Y, NAMES, META = mimic_iv.build_cohort(CFG)
VIEWS = {k: np.asarray(v) for k, v in META["views"].items()}
print(f"[v2] cohort n={len(Y)} features={X_RAW.shape[1]} prevalence={Y.mean():.4f} "
      f"views={ {k: len(v) for k, v in VIEWS.items()} } ({time.time()-t0:.0f}s)", flush=True)

## Run everything, per seed

In [16]:
from xgboost import XGBClassifier
from src import arch4r
from src.encoder_mv import fit_view_preprocessors, observed_fraction
from src.local_push import make_checkpoints

Y_NP = Y.astype(np.int64)
ALL = {}


def rep(name, row):
    print(f"[v2][seed {SEED}] {name:42s} " + " ".join(
        f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items()), flush=True)
    ALL.setdefault(str(SEED), {})[name] = row
    with open(f"{OUT}/results_v2.json", "w") as f:
        json.dump(ALL, f, indent=1)


def exit_report(outs, mv, te, extra=None):
    """Per-exit test AUROC; `best_exit` chosen on model_val, never on test."""
    va = [auroc(Y_NP[mv], o[0]) for o in outs]
    ta = [auroc(Y_NP[te], o[1]) for o in outs]
    b = int(np.argmax(va))
    row = {"test_auroc_last": ta[-1], "test_auroc_bestval": ta[b], "best_exit": b,
           "val_auroc_last": va[-1], "test_auprc_last": auprc(Y_NP[te], outs[-1][1]),
           "test_auroc_per_exit": [round(t, 4) for t in ta]}
    row.update(extra or {})
    return row


for SEED in SEEDS:
    set_seed(SEED)
    train, val, test = make_splits(Y_NP, SEED)
    mv, gsel, gcert = split_validation_3way(val, Y_NP, SEED)
    tr_idx, mv_idx, te_idx = np.flatnonzero(train), np.flatnonzero(mv), np.flatnonzero(test)
    print(f"[v2][seed {SEED}] train={len(tr_idx)} model_val={len(mv_idx)} test={len(te_idx)}", flush=True)
    y_t = torch.tensor(Y_NP, dtype=torch.float32, device=DEV)

    # ---- 1. XGBoost references ------------------------------------------------
    Xi = X_RAW.astype(np.float32).copy()
    miss = np.isnan(Xi)
    med = np.nan_to_num(np.nanmedian(Xi[train], 0))
    Xi = np.where(miss, med, Xi)
    mu, sd = Xi[train].mean(0), Xi[train].std(0)
    Xi = (Xi - mu) / np.where(sd < 1e-6, 1.0, sd)
    Xi = np.concatenate([Xi, miss[:, miss[train].any(0)].astype(np.float32)], 1)
    pos = (Y_NP[train] == 1).sum()
    clf = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, scale_pos_weight=float((Y_NP[train] == 0).sum() / pos),
                        eval_metric="logloss", random_state=SEED, n_jobs=4, tree_method="hist",
                        device="cuda" if DEV.type == "cuda" else "cpu")
    clf.fit(Xi[train], Y_NP[train])
    rep("xgb_plain_phase1_config", {"test_auroc": auroc(Y_NP[test], clf.predict_proba(Xi[test])[:, 1]),
                                    "test_auprc": auprc(Y_NP[test], clf.predict_proba(Xi[test])[:, 1])})
    clf = XGBClassifier(n_estimators=4000, max_depth=5, learning_rate=0.02, subsample=0.8,
                        colsample_bytree=0.5, min_child_weight=2, reg_lambda=2.0,
                        eval_metric="auc", early_stopping_rounds=300, random_state=SEED, n_jobs=4,
                        tree_method="hist", device="cuda" if DEV.type == "cuda" else "cpu")
    clf.fit(X_RAW[train], Y_NP[train], eval_set=[(X_RAW[mv], Y_NP[mv])], verbose=False)
    p = clf.predict_proba(X_RAW[test])[:, 1]
    rep("xgb_tuned_nativeNaN_earlystop", {"test_auroc": auroc(Y_NP[test], p), "test_auprc": auprc(Y_NP[test], p),
                                          "best_iteration": int(clf.best_iteration)})

    # ---- 2. corrected Arch4R encoder_only baseline (original model + preprocessing)
    raw_views = {v: X_RAW[:, idx] for v, idx in VIEWS.items()}
    pre = fit_view_preprocessors(raw_views, train)
    vx = pre.transform(raw_views)
    vo = observed_fraction(raw_views)
    cfg_a = {"fusion_dim": 64, "stage_hidden": 32}
    for proto in ("phase1_fullbatch_5ep", "minibatch_30ep", "minibatch_earlystop"):
        set_seed(SEED)
        m4 = arch4r.Arch4RModel({v: a.shape[1] for v, a in vx.items()}, cfg_a, "encoder_only").to(DEV)
        ck = make_checkpoints()
        VX = {v: torch.tensor(a, device=DEV) for v, a in vx.items()}
        VO = {v: torch.tensor(a, dtype=torch.float32, device=DEV) for v, a in vo.items()}
        fwd4 = lambda b, train=True: m4({v: t[b] for v, t in VX.items()},
                                        {v: t[b] for v, t in VO.items()}, None, ck)["stage_logits"][-1:]
        val4 = lambda: [auroc(Y_NP[mv_idx], predict_rows(fwd4, m4, mv_idx)[-1])]
        if proto == "phase1_fullbatch_5ep":   # exactly what phase 1 did: 1 Adam step/epoch
            opt = torch.optim.Adam(m4.parameters(), lr=1e-3)
            for _ in range(5):
                m4.train(); out = fwd4(tr_idx)
                loss = F.binary_cross_entropy_with_logits(out[0], y_t[tr_idx])
                opt.zero_grad(); loss.backward(); opt.step()
            info = {}
        elif proto == "minibatch_30ep":
            # phase-1's own optimiser (Adam, no weight decay) and fixed 30-epoch
            # budget, LAST-epoch weights -- only the batching differs.
            m4_opt_wd = 0.0
            info = fit_minibatch(m4, fwd4, tr_idx, y_t, val4, wd=m4_opt_wd, max_ep=30,
                                 patience=10**9, seed=SEED, restore_best=False)
        else:
            info = fit_minibatch(m4, fwd4, tr_idx, y_t, val4, seed=SEED)
        pm = predict_rows(fwd4, m4, mv_idx)[-1]; pt = predict_rows(fwd4, m4, te_idx)[-1]
        rep(f"arch4r_encoder_only[{proto}]", {"test_auroc": auroc(Y_NP[te_idx], pt),
                                              "val_auroc": auroc(Y_NP[mv_idx], pm), **info})

    # ---- 3. encoder preprocessing ablation --------------------------------------
    abl = {}
    for kind in ("iqr", "pct", "quantile"):
        for ind in (False, True):
            set_seed(SEED)
            Xp, sl = prep_views(X_RAW, VIEWS, train, kind, ind)
            Xt = torch.tensor(Xp, device=DEV)
            m0 = EncoderOnly(sl).to(DEV)
            fwd = lambda b, train=True: m0(Xt[b])
            vf = lambda: [auroc(Y_NP[mv_idx], predict_rows(fwd, m0, mv_idx)[-1])]
            info = fit_minibatch(m0, fwd, tr_idx, y_t, vf, seed=SEED)
            outs = list(zip(predict_rows(fwd, m0, mv_idx), predict_rows(fwd, m0, te_idx)))
            abl[(kind, ind)] = info["val_auroc"]
            rep(f"encoder[{kind},ind={int(ind)}]", exit_report(outs, mv_idx, te_idx, info))
    BEST_PREP = max(abl, key=abl.get)
    print(f"[v2][seed {SEED}] best preprocessing on model_val: {BEST_PREP}", flush=True)
    Xp, SL = prep_views(X_RAW, VIEWS, train, *BEST_PREP)
    Xt = torch.tensor(Xp, device=DEV)

    # ---- 4. label-homophily diagnostics (train-train edges only) -----------------
    with torch.no_grad():
        Z = F.normalize(Xt, dim=1)
        ytr = y_t[tr_idx]
        prev = float(ytr.mean())
        for kk in (16, 64):
            nb = []
            for i in range(0, len(tr_idx), 4096):
                sim = Z[tr_idx[i:i + 4096]] @ Z[tr_idx].T
                sim[torch.arange(sim.shape[0], device=DEV), torch.arange(i, i + sim.shape[0], device=DEV)] = -2
                nb.append(sim.topk(kk, 1).indices)
            nb = torch.cat(nb)
            rate = ytr[nb].mean(1)
            same = (ytr[:, None] == ytr[nb]).float().mean()
            h0 = prev ** 2 + (1 - prev) ** 2
            m11 = float(ytr[nb][ytr == 1].mean()); m00 = float(1 - ytr[nb][ytr == 0].mean())
            sim_mv = Z[mv_idx] @ Z[tr_idx].T
            rate_mv = ytr[sim_mv.topk(kk, 1).indices].mean(1).cpu().numpy()
            rep(f"homophily[feature_cosine_knn,k={kk}]", {
                "edge_h": float(same), "adj_h": float((same - h0) / (1 - h0)), "M11": m11, "M00": m00,
                "loo_label_rate_auroc_train": auroc(Y_NP[tr_idx], rate.cpu().numpy()),
                "label_rate_auroc_model_val": auroc(Y_NP[mv_idx], rate_mv)})

    # ---- 5. G1 retrieval ---------------------------------------------------------
    pool_pos = torch.full((len(Y_NP),), -1, dtype=torch.long, device=DEV)
    pool_pos[torch.as_tensor(tr_idx, device=DEV)] = torch.arange(len(tr_idx), device=DEV)
    x_pool, y_pool = Xt[tr_idx], y_t[tr_idx].long()
    for m in (32, 96):
        set_seed(SEED)
        g1 = Retrieval(SL, m=m).to(DEV)
        fwd = lambda b, train=True: g1(Xt[b], pool_pos[torch.as_tensor(b, device=DEV)] if train else None,
                                       x_pool, y_pool)
        vf = lambda: [auroc(Y_NP[mv_idx], predict_rows(fwd, g1, mv_idx, bs=1024)[-1])]
        info = fit_minibatch(g1, fwd, tr_idx, y_t, vf, lr=3e-4, wd=1e-5, bs=256, patience=16, seed=SEED)
        outs = list(zip(predict_rows(fwd, g1, mv_idx, bs=1024), predict_rows(fwd, g1, te_idx, bs=1024)))
        rep(f"G1_retrieval[m={m}]", exit_report(outs, mv_idx, te_idx, info))

    # ---- 6. G2 masked-label PPR ---------------------------------------------------
    for keep in (0.5, 0.75):
        set_seed(SEED)
        g2 = LabelPPR(SL).to(DEV)
        info = fit_labelppr(g2, Xt, y_t, tr_idx, mv_idx, keep=keep, seed=SEED)
        outs = list(zip(labelppr_predict(g2, Xt, y_t, tr_idx, mv_idx),
                        labelppr_predict(g2, Xt, y_t, tr_idx, te_idx)))
        rep(f"G2_labelPPR[keep={keep}]", exit_report(outs, mv_idx, te_idx,
                                                     info | {"tau": float(g2.log_tau.exp())}))

print("[v2] DONE", flush=True)
print(json.dumps(ALL, indent=1))

/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


[v2] device=cuda torch=2.10.0+cu128
[v2] smoke test OK


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (256) is greater than the total number of samples (200). n_quantiles is set to n_samples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (256) is greater than the total number of samples (200). n_quantiles is set to n_samples.
  warnings.warn(

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (256) is greater than the total number of samples (200). n_quantiles is set to n_samples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (256) is greater than the total number of samples (200). n_quantiles is set to n_samples.
  warnings.warn(


[v2] MIMIC root: /kaggle/input/datasets/akshaybe/updated-mimic-iv
[v2] cohort signature: 97de93de81
[mimic] building cohort from /kaggle/input/datasets/akshaybe/updated-mimic-iv/mimic-iv-3.1
  icu stays                      94,458
  first stay per patient         65,366
  adult                          65,366
  los >= 28h                     46,776
  died after window              46,705
  discharged after window        46,651
  [demographics] built 33 cols in 0.3s -> /kaggle/working/processed/block_63737e3178_demographics.parquet
  [comorbidity] built 19 cols in 5.7s -> /kaggle/working/processed/block_63737e3178_comorbidity.parquet
  [lab] duckdb scan of hosp/labevents.csv ...
  [labs] built 168 cols in 129.9s -> /kaggle/working/processed/block_63737e3178_labs.parquet
  [vit] duckdb scan of icu/chartevents.csv ...
  [vitals] built 84 cols in 359.5s -> /kaggle/working/processed/block_63737e3178_vitals.parquet
  [tx] duckdb scan of icu/inputevents.csv ...
  [tx] duckdb scan of icu/proce

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [06:07:18] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [06:07:18] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the devi

[v2][seed 42] xgb_tuned_nativeNaN_earlystop              test_auroc=0.9018 test_auprc=0.5928 best_iteration=656
[v2][seed 42] arch4r_encoder_only[phase1_fullbatch_5ep]  test_auroc=0.7387 val_auroc=0.7673
[v2][seed 42] arch4r_encoder_only[minibatch_30ep]        test_auroc=0.8531 val_auroc=0.8983 best_epoch=6 epochs_run=30 restored_best=False train_s=4.7455
[v2][seed 42] arch4r_encoder_only[minibatch_earlystop]   test_auroc=0.8866 val_auroc=0.8983 best_epoch=6 epochs_run=27 restored_best=True train_s=4.2719
[v2][seed 42] encoder[iqr,ind=0]                         test_auroc_last=0.8924 test_auroc_bestval=0.8924 best_exit=0 val_auroc_last=0.9017 test_auprc_last=0.5601 test_auroc_per_exit=[0.8924] val_auroc=0.9017 best_epoch=3 epochs_run=24 restored_best=True train_s=3.0156
[v2][seed 42] encoder[iqr,ind=1]                         test_auroc_last=0.8981 test_auroc_bestval=0.8981 best_exit=0 val_auroc_last=0.9019 test_auprc_last=0.5733 test_auroc_per_exit=[0.8981] val_auroc=0.9019 best_epoch

/tmp/ipykernel_22/2463680247.py:161: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  info | {"tau": float(g2.log_tau.exp())}))

/tmp/ipykernel_22/2463680247.py:161: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  info | {"tau": float(g2.log_tau.exp())}))


[v2][seed 42] G2_labelPPR[keep=0.75]                     test_auroc_last=0.8941 test_auroc_bestval=0.8943 best_exit=0 val_auroc_last=0.8961 test_auprc_last=0.5569 test_auroc_per_exit=[0.8943, 0.8941, 0.8943, 0.8938, 0.8943, 0.8941] val_auroc=0.8961 best_step=60 steps_run=560 train_s=75.1141 tau=0.0987
[v2] DONE
{
 "42": {
  "xgb_plain_phase1_config": {
   "test_auroc": 0.8958150139227521,
   "test_auprc": 0.5545743072592642
  },
  "xgb_tuned_nativeNaN_earlystop": {
   "test_auroc": 0.9018004649799486,
   "test_auprc": 0.592753692549146,
   "best_iteration": 656
  },
  "arch4r_encoder_only[phase1_fullbatch_5ep]": {
   "test_auroc": 0.7387085814780767,
   "val_auroc": 0.7672951709169831
  },
  "arch4r_encoder_only[minibatch_30ep]": {
   "test_auroc": 0.8531149659592043,
   "val_auroc": 0.8983263074418799,
   "best_epoch": 6,
   "epochs_run": 30,
   "restored_best": false,
   "train_s": 4.745508670806885
  },
  "arch4r_encoder_only[minibatch_earlystop]": {
   "test_auroc": 0.8865612134777

/usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
  cells[i][c] = re.sub('\\\\\|', '|', cell)
/usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
  text = re.sub(r'_', '\_', text) # Escape underscores in display text
[NbConvertApp] Converting notebook __notebook__.ipynb to notebook
[NbConvertApp] Writing 222767 bytes to __notebook__.ipynb
[NbConvertApp] Converting notebook __notebook__.ipynb to html
[NbConvertApp] Writing 1006837 bytes to __results__.html
